# BurnWise — Team merged pipeline

ฐาน: notebook ของเพื่อน (Burn/No Burn, FTW, GISTDA) รวมกับ DOPA, ถนน GraphML และรายได้ จปฐ. ของทีม

รันส่วน A–K เพื่อสร้างผลหลักก่อน ส่วน O2 เป็นการเปรียบเทียบกับผลิตภัณฑ์ GISTDA ภายนอก ต้องมี reference ครบช่วงเวลาและไฟล์ raster จาก GEE จึงประเมินได้ ไม่ใช่ accuracy ต่อ ground truth ภาคสนาม

ไฟล์หลักที่ต้องวางข้าง notebook:
- `2025_N15E100.parquet`
- `burnwise_roads.graphml`
- `thatako_socioeconomic_data_2568.csv`
- `tha_tako_tambons_dopa.geojson` (ถ้าไม่มีจะดึง DOPA ด้วย requests)

ข้อมูลเพิ่มสำหรับ O2: Burned_S2 พ.ย. 2025, ธ.ค. 2025, ม.ค. 2026 พร้อม shapefile components และ `burnwise_evaluation_grid.tif` ที่ export จากเซลล์ O2

ค่าความครอบคลุม 80%, เกณฑ์แปลง 10%, Tier 50%, น้ำหนัก Access Gap 50/30/20 เป็นเกณฑ์ทดลอง ยังไม่ได้ optimize/ยืนยันภาคสนาม จำนวนภาพ pre/post ต้องตรวจแยกจากค่าความครอบคลุมภาพ

ไม่มีผลรันเดิมแนบในไฟล์รวม เพื่อไม่ให้ผลเก่าปะปนกับสูตรใหม่

## ลำดับรันฉบับรวม
- ส่วนหลักมี **11 Code cells A–K** รันตามหัวข้อจากบนลงล่าง ไม่มีเซลล์ Q/mock/P เก่าปะปน
- ส่วน O2 มี **5 Code cells** สำหรับการเปรียบเทียบภายนอก รันเมื่อเตรียม reference ครบแล้ว
- ขั้น F บันทึก checkpoint เมื่อสถิติรายแปลงเสร็จ หากสูตรและข้อมูลต้นทางเดิมจะโหลด checkpoint เมื่อรันครั้งถัดไป
- เมื่อต้องปรับเพียงสูตรรายได้/มาตรการ ใช้ตัวแปรจากส่วน F–G ที่รันแล้ว แล้วรัน H–K ต่อ ไม่ต้องดึงภาพซ้ำ
- การเปลี่ยนวัน/threshold/NDVI/AOI/ไฟล์ FTW ทำให้ checkpoint เปลี่ยนชื่ออัตโนมัติ ต้องคำนวณสถิติใหม่
- ขั้น O2 ตรวจเดือนที่ขาดก่อนประเมิน: ถ้ามีแค่มกราคมยังทำผลหลักได้ แต่ยังไม่ประเมินครบพฤศจิกายน–มกราคม
- ข้อมูลรายได้และถนนต้องมีจริง หากขาด notebook จะแจ้ง ไม่เติมค่าจำลองให้

**ตรวจไฟล์รวม:** ตรวจ syntax และทดสอบเงื่อนไขสถานะด้วยข้อมูลสังเคราะห์แล้ว ยังไม่ได้รัน Earth Engine หรือไฟล์ GISTDA/FTW จริงในเครื่องนี้

## ส่วนแสดงกราฟที่เพิ่ม
- **Code cell 11 / K** อยู่หลัง J: กราฟ 6 แบบจากผลจริง ไม่มีการดึง GEE เพิ่ม
- หากมี `master_plots` และ `df_tambon_overview` ใน memory อยู่แล้ว รัน K ได้เลย
- ถ้าเปิด session ใหม่และมี CSV ของ J ให้ติดตั้ง Plotly แล้วรัน K ได้โดยอ่าน CSV
- กราฟรายตำบล สถานะ/coverage/เหตุผล Unknown และถนนใช้ข้อมูลครบ ส่วน scatter แสดงไม่เกิน 6,000 แปลงพร้อมจำนวนกำกับ
- HTML รวมอยู่ที่ `burnwise_charts/burnwise_charts.html` เปิดได้ออฟไลน์ พร้อม JSON และตารางฐานกราฟในโฟลเดอร์เดียวกัน
- ส่วน O2 เดิมเลื่อนไปเป็น Code cells **12–16**; ไม่ต้องรัน O2 เพื่อดูกราฟหลัก
- กราฟเป็นผลสำรวจ ยังไม่ใช่ ground truth/คะแนนความแม่นยำ และไม่ได้ปรับ schema ของแอป Streamlit เดิมให้อัตโนมัติ


## A · ติดตั้งและเชื่อมต่อ Earth Engine

In [ ]:
import ee
import geemap

ee.Authenticate()

# ใส่ Google Cloud project ที่ผูกกับ Earth Engine
PROJECT_ID =
ee.Initialize(project=PROJECT_ID)

## B · ขอบเขต DOPA และ grid กลาง

In [ ]:
import json, os, requests
from pathlib import Path
geojson_path = 'tha_tako_tambons_dopa.geojson'
if not Path(geojson_path).is_file():
    url='https://ngis.go.th/arcgis/rest/services/Hosted/ขอบเขตตำบล_DOPA/FeatureServer/0/query'
    response=requests.get(url,params={
        'where':"amphoe_t = 'อ.ท่าตะโก' AND changwat_t = 'จ.นครสวรรค์'",
        'outFields':'*','returnGeometry':'true','outSR':4326,'f':'geojson'
    },timeout=90)
    response.raise_for_status()
    data=response.json()
    if 'error' in data or not data.get('features'):
        raise ValueError(f'DOPA ไม่คืนข้อมูลตำบล: {data}')
    if data.get('exceededTransferLimit'):
        raise ValueError('DOPA คืนข้อมูลไม่ครบ ต้องใช้ pagination')
    for f in data['features']:
        p=f['properties'];p['tambon_id']=str(p['tambon_id'])
        p['tambon_name']=p['tambon_t']
    Path(geojson_path).write_text(json.dumps(data,ensure_ascii=False),encoding='utf-8')
with open(geojson_path,encoding='utf-8') as f: tambons_geojson=json.load(f)
# CRS ที่เป็น null ไม่ใช่ declaration ที่ Earth Engine ยอมรับ
# ลบเฉพาะค่า null; เก็บ CRS ที่ระบุไว้จริงและพิกัดเดิม
if tambons_geojson.get('crs', 'missing') is None:
    tambons_geojson.pop('crs', None)
for feat in tambons_geojson['features']:
    geometry = feat.get('geometry')
    if geometry is None:
        raise ValueError(f"ตำบล {feat.get('properties', {}).get('tambon_name', '?')} ไม่มี geometry")
    if geometry.get('crs', 'missing') is None:
        geometry.pop('crs', None)
assert len(tambons_geojson['features'])==10, 'ต้องมี 10 ตำบลในท่าตะโก'
assert len({str(f['properties']['tambon_id']) for f in tambons_geojson['features']})==10
print('ขอบเขต DOPA:',len(tambons_geojson['features']),'ตำบล')

ee_features=[]
for feat in tambons_geojson['features']:
    p=feat['properties']
    ee_features.append(ee.Feature(ee.Geometry(feat['geometry']),{
        'tambon_id':str(p['tambon_id']), 'tambon_name':p['tambon_name']}))
tambons=ee.FeatureCollection(ee_features)
aoi=tambons.geometry()
# ตาราง/raster validation ใช้ grid 10 ม. UTM47N เดียวกัน
EVAL_CRS='EPSG:32647'
EVAL_TRANSFORM=[10,0,0,0,-10,0]
MIN_VALID_OBSERVATION_PCT=80.0
print('AOI พร้อม')

## C · ช่วงเวลาและ Sentinel-2 — ตรวจจำนวนภาพก่อนใช้ผล

In [ ]:
# ⚠️ แก้ตามรีวิว (ข้อ 4): เดิมช่วงก่อน 6 เดือน / หลัง 4 เดือน ยาวเกินไปและไม่สมดุลกัน
# ปรับให้แคบลงและสมมาตรกัน (3 เดือนเท่ากันทั้งคู่) ลดโอกาสปนเปื้อนจากการเปลี่ยนพืช/ฤดูอื่น
pre_start,  pre_end  = '2025-08-01', '2025-11-01'    # filterDate ใช้วันสิ้นสุดแบบไม่รวมวันนั้น: ครอบคลุม 1 ส.ค.–31 ต.ค. 2025
# ประเมินเฉพาะเดือนมกราคม 2569
post_start, post_end = '2026-01-01', '2026-02-01'   # ครอบคลุม 1 พ.ย. 2025–31 ม.ค. 2026; ใช้เป็นช่วงเก็บเกี่ยว/หลังเก็บเกี่ยวสำหรับ FIRMS

print(f'ช่วงก่อนเก็บเกี่ยว: {pre_start} ถึง {pre_end} (วันสิ้นสุดไม่รวม; ถึง 31 ต.ค. 2025)')
print(f'ช่วงหลังเก็บเกี่ยว/ช่วงตรวจ FIRMS: {post_start} ถึง {post_end} (วันสิ้นสุดไม่รวม; ถึง 31 ม.ค. 2026)')
print('หมายเหตุ: ถ้าจำนวนภาพ (เซลล์ถัดไป) ยังน้อยหรือไม่สมดุลมาก ให้ขยับช่วงนี้อีกครั้งแล้วรันใหม่')


def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

pre_collection  = s2.filterDate(pre_start, pre_end).map(mask_s2_clouds)
post_collection = s2.filterDate(post_start, post_end).map(mask_s2_clouds)

n_pre_images  = pre_collection.size().getInfo()
n_post_images = post_collection.size().getInfo()
print('จำนวนภาพช่วงก่อนเก็บเกี่ยว:', n_pre_images)
print('จำนวนภาพช่วงหลังเก็บเกี่ยว:', n_post_images)

# ⚠️ เพิ่มตามรีวิว (ข้อ 4): เตือนถ้าจำนวนภาพสองช่วงต่างกันมาก
# เพราะ median composite ที่มาจากภาพน้อยจะ noise สูงกว่า ทำให้ dNBR ทั้งสองฝั่งไม่ได้มาตรฐานเดียวกัน
if min(n_pre_images, n_post_images) == 0:
    # ⚠️ แก้ตาม Stage 1 (C): เดิม print() เฉยๆ แล้วปล่อยให้รันต่อด้วย composite ว่าง
    # ซึ่งทำให้ dNBR/threshold/ผลทั้งหมดพังแบบไม่รู้ตัว — เปลี่ยนเป็น raise หยุดทันที
    raise ValueError(
        f'ช่วงใดช่วงหนึ่งไม่มีภาพเลย (pre={n_pre_images}, post={n_post_images}) — '
        f'ต้องขยายช่วงวันที่ pre_start/pre_end หรือ post_start/post_end แล้วรันใหม่ตั้งแต่เซลล์นี้ '
        f'ห้ามรันต่อ เพราะ median composite จากภาพ 0 ภาพจะเป็นค่าว่างล้วน ทำให้ dNBR/threshold ที่คำนวณต่อผิดทั้งหมด'
    )
else:
    ratio = max(n_pre_images, n_post_images) / min(n_pre_images, n_post_images)
    print(f'อัตราส่วนจำนวนภาพ pre:post = {ratio:.1f} เท่า')
    if ratio > 3:
        print(f'⚠️ จำนวนภาพสองช่วงต่างกันเกิน 3 เท่า — median composite ฝั่งที่มีภาพน้อยจะมี noise สูงกว่ามาก')
        print('   พิจารณาขยายช่วงวันที่ฝั่งที่มีภาพน้อยออกไปอีก หรือระบุข้อจำกัดนี้ตอนรายงานผลตรงๆ')
    else:
        print('✅ จำนวนภาพสองช่วงสมดุลกันพอสมควร')


pre_composite  = pre_collection.median().clip(aoi)
post_composite = post_collection.median().clip(aoi)


nbr_pre  = pre_composite.normalizedDifference(['B8', 'B12']).rename('NBR_pre')
nbr_post = post_composite.normalizedDifference(['B8', 'B12']).rename('NBR_post')
dNBR = nbr_pre.subtract(nbr_post).rename('dNBR')

ndvi_pre  = pre_composite.normalizedDifference(['B8', 'B4']).rename('NDVI_pre')
ndvi_post = post_composite.normalizedDifference(['B8', 'B4']).rename('NDVI_post')

worldcover = ee.ImageCollection('ESA/WorldCover/v100').first()
CROPLAND_CLASS = 40  # ESA WorldCover v100: 40 = Cropland
cropland_mask = worldcover.select('Map').eq(CROPLAND_CLASS)

dNBR_crop = dNBR.updateMask(cropland_mask).clip(aoi)

print('คำนวณ NDVI/NBR/dNBR และ mask พื้นที่เกษตรเรียบร้อย')
print('พื้นที่เกษตรจะสรุปจาก stats_image ในขั้น E')


## D · Threshold baseline, hotspot calibration และ test proxy

In [ ]:
histogram=dNBR_crop.reduceRegion(reducer=ee.Reducer.histogram(255,0.005),
    geometry=aoi,scale=10,maxPixels=1e9,tileScale=16).get('dNBR').getInfo()
import numpy as np
if not histogram:raise ValueError('ไม่มี histogram dNBR ใช้คำนวณ threshold')
counts=np.asarray(histogram['histogram'],dtype=float)
means=np.asarray(histogram['bucketMeans'],dtype=float)
if len(counts)<2 or counts.sum()==0:raise ValueError('histogram มีข้อมูลไม่พอ')
w=np.cumsum(counts)[:-1]; total=counts.sum()
cs=np.cumsum(counts*means)[:-1]; ts=(counts*means).sum()
ok=(w>0)&(w<total)
bss=np.full(len(w),-np.inf)
bss[ok]=w[ok]*(total-w[ok])*((cs[ok]/w[ok])-((ts-cs[ok])/(total-w[ok])))**2
if not ok.any():raise ValueError('histogram แยกสองกลุ่มไม่ได้')
cut=int(np.argmax(bss));otsu_value=float((means[cut]+means[cut+1])/2)
print('Otsu baseline:',otsu_value)


# ============================================================
# F) VIIRS 375 m Active Fire / FIRMS — confidence filtering
# ============================================================
# ใช้ VIIRS 375 m จาก NASA FIRMS แทน FIRMS raster 1 km เดิม
# รวม Suomi-NPP + NOAA-20 และคัดเฉพาะ confidence >= 1
# confidence: 0=low, 1=nominal, 2=high
# อ้างอิง: NASA/Google Earth Engine VIIRS 375 m Active Fire

VIIRS_MIN_CONFIDENCE = 1

viirs_snpp = (
    ee.ImageCollection('NASA/LANCE/SNPP_VIIRS/C2')
    .filterDate(post_start, post_end)
    .filterBounds(aoi)
)

viirs_noaa20 = (
    ee.ImageCollection('NASA/LANCE/NOAA20_VIIRS/C2')
    .filterDate(post_start, post_end)
    .filterBounds(aoi)
)

viirs = viirs_snpp.merge(viirs_noaa20)

# confidence >= 1 = nominal/high confidence
viirs_high_conf = viirs.map(
    lambda img: img.updateMask(img.select('confidence').gte(VIIRS_MIN_CONFIDENCE))
)

# รวมทุก acquisition เป็น fire mask 375 m
viirs_fire_mask = (
    viirs_high_conf
    .select('Bright_ti4')
    .max()
    .gt(0)
    .selfMask()
    .clip(aoi)
    .rename('viirs_fire')
)

# แปลง fire pixels เป็น centroid points สำหรับ calibration/validation
fire_points = viirs_fire_mask.reduceToVectors(
    geometry=aoi,
    scale=375,
    geometryType='centroid',
    labelProperty='fire',
    maxPixels=1e9
)

# ตรวจ band/ชนิดข้อมูลจริงของ collection ก่อนเชื่อ filter confidence
# ⚠️ แก้ตาม Stage 1 (D): เช็ค .size() ก่อนเรียก .first() เสมอ — ถ้า viirs ว่าง (ไม่มีภาพผ่านช่วง/พื้นที่นี้เลย)
# .first().getInfo() หรือ .bandNames().getInfo() จะโยน EEException ทันที ทำให้เซลล์ทั้งเซลล์พังไปด้วย
_viirs_n = viirs.size().getInfo()
if _viirs_n == 0:
    print('⚠️ ไม่มีภาพ VIIRS (Suomi-NPP + NOAA-20) เลยในช่วง/พื้นที่นี้ — ข้ามการเช็ค band แล้วไปต่อด้วย fire_points ว่าง')
else:
    _first = viirs.first()
    print('VIIRS bands:', _first.bandNames().getInfo())
    print('confidence band type:', _first.select('confidence').bandTypes().getInfo() if 'confidence' in _first.bandNames().getInfo() else 'ไม่พบ band confidence')

n_fire = fire_points.size().getInfo()
print('VIIRS 375 m hotspot (confidence >= 1):', n_fire)
print('Suomi-NPP images:', viirs_snpp.size().getInfo())
print('NOAA-20 images:', viirs_noaa20.size().getInfo())

# (ค่าเฉลี่ยจากทุกจุดนี้ใช้เพื่อดูภาพรวมเท่านั้น — THRESHOLD จริงมาจากชุด Calibration ใน F.1)
mean_fire_dnbr = None
if n_fire > 0:
    fire_dnbr_fc = dNBR_crop.reduceRegions(
        collection=fire_points,
        reducer=ee.Reducer.mean(),
        scale=10,
        tileScale=4
    )
    fire_dnbr_values = [v for v in fire_dnbr_fc.aggregate_array('mean').getInfo() if v is not None]
    if fire_dnbr_values:
        import statistics
        mean_fire_dnbr = statistics.mean(fire_dnbr_values)
        print(f'dNBR เฉลี่ยที่ VIIRS hotspot: {mean_fire_dnbr:.4f}  (Otsu = {otsu_value:.4f})')
        if mean_fire_dnbr > otsu_value:
            print('>>> VIIRS hotspot มี dNBR สูงกว่า Otsu <<<')
        else:
            print('>>> ⚠️ VIIRS hotspot มี dNBR ไม่สูงกว่า Otsu ชัดเจน <<<')
else:
    print('ไม่พบ VIIRS hotspot confidence >= 1 ใน AOI ช่วงนี้ — ใช้ Otsu แทน')


import random
random.seed(42)

fire_ids = fire_points.aggregate_array('system:index').getInfo()
random.shuffle(fire_ids)
split_at = max(1, int(len(fire_ids) * 0.7))
calib_ids = fire_ids[:split_at]
test_ids  = fire_ids[split_at:]
print(f'จุด FIRMS ทั้งหมด: {len(fire_ids)} -> Calibration: {len(calib_ids)}, Test (แยกตัวอย่าง): {len(test_ids)}')

calib_points = fire_points.filter(ee.Filter.inList('system:index', calib_ids))
test_points  = fire_points.filter(ee.Filter.inList('system:index', test_ids))

# คำนวณ threshold จากชุด Calibration เท่านั้น (ไม่แตะชุด Test เลย)
calib_dnbr_fc = dNBR_crop.reduceRegions(collection=calib_points, reducer=ee.Reducer.mean(), scale=10)
calib_dnbr_values = [v for v in calib_dnbr_fc.aggregate_array('mean').getInfo() if v is not None]
import statistics
mean_fire_dnbr_calib = statistics.mean(calib_dnbr_values) if calib_dnbr_values else None
print(f'Threshold จากชุด Calibration เท่านั้น: {mean_fire_dnbr_calib}')


# Validate เฉพาะช่วงเก็บเกี่ยว/หลังเก็บเกี่ยวที่กำหนดไว้ใน post_start:post_end
# FIRMS เป็น positive reference; ผลนี้เป็น recall proxy ไม่ใช่ full accuracy
if mean_fire_dnbr_calib is not None and test_points.size().getInfo() > 0:

    test_burn_mask = (
        dNBR_crop.gt(mean_fire_dnbr_calib)
        .And(ndvi_pre.gt(0.3))
        .And(ndvi_post.lt(0.3))
        .And(cropland_mask)
        .rename('burn_test_harvest')
    )

    test_buffers = test_points.map(lambda f: f.buffer(500))

    test_burn_check = test_burn_mask.reduceRegions(
        collection=test_buffers,
        reducer=ee.Reducer.mean(),
        scale=10,
        tileScale=4
    )

    test_results = test_burn_check.aggregate_array('mean').getInfo()
    test_results = [r for r in test_results if r is not None]

    n_test = len(test_results)
    MIN_BURN_FRACTION = 0.10
    n_detected = sum(1 for r in test_results if r >= MIN_BURN_FRACTION)
    recall_on_test = n_detected / n_test if n_test > 0 else None

    print(f'ช่วงภาพ Pre: {pre_start} ถึง {pre_end} (วันสิ้นสุดไม่รวม)')
    print(f'ช่วงภาพ Post/FIRMS: {post_start} ถึง {post_end} (วันสิ้นสุดไม่รวม)')
    print(f'Test FIRMS ที่มีข้อมูล raster ใช้ได้: {n_test}')
    print(
        f'Test ที่พบ burn + harvest อย่างน้อย '
        f'{MIN_BURN_FRACTION:.0%} ใน buffer 500 m: {n_detected}/{n_test}'
    )
    print(f'Recall proxy ในช่วงเก็บเกี่ยว: {recall_on_test}')
    print('ข้อจำกัด: FIRMS เป็น positive reference เท่านั้น จึงยังคำนวณ TN/FP/Precision/Accuracy ไม่ได้')

else:
    print('ไม่มีข้อมูล Test เพียงพอ หรือคำนวณ threshold จาก Calibration ไม่ได้')


# เลือก THRESHOLD ที่ใช้จริง — ใช้เฉพาะชุด Calibration (F.1) เพื่อให้ชุด Test เป็นอิสระจริง
# ถ้าไม่มีจุด Calibration ใช้ให้ถอยไปใช้ Otsu
if mean_fire_dnbr_calib is not None:
    THRESHOLD = mean_fire_dnbr_calib
    THRESHOLD_SOURCE = (f'ค่าเฉลี่ย dNBR ที่จุดความร้อน VIIRS ชุด Calibration ({len(calib_dnbr_values)} จุด) '
                        f'— ชุด Test {len(test_ids)} จุดไม่ถูกใช้กำหนดค่า')
else:
    THRESHOLD = otsu_value
    THRESHOLD_SOURCE = 'วิธี Otsu (ไม่มีจุดความร้อน VIIRS สำหรับ calibration ในพื้นที่นี้)'

print(f'>>> THRESHOLD ที่ใช้จริง = {THRESHOLD:.4f}  (dNBR > ค่านี้ = ถือว่าเผา) <<<')
print(f'>>> อ้างอิงจาก: {THRESHOLD_SOURCE} <<<')


## E · Burn/No Burn, ความครอบคลุมภาพ และสรุปตำบล

In [ ]:
NDVI_PRE_MIN=0.30; NDVI_POST_MAX=0.30
valid_crop_mask=(cropland_mask.unmask(0)
    .And(dNBR_crop.mask().unmask(0))
    .And(ndvi_pre.mask().unmask(0))
    .And(ndvi_post.mask().unmask(0))).rename('valid_crop')
burn_mask=(dNBR_crop.gt(THRESHOLD)
    .And(ndvi_pre.gt(NDVI_PRE_MIN)).And(ndvi_post.lt(NDVI_POST_MAX))
    .updateMask(valid_crop_mask).rename('burn'))
no_burn_mask=burn_mask.Not().updateMask(valid_crop_mask).rename('no_burn')
VIIRS_CONFIRM_BUFFER_M=250
viirs_fire_buffer=fire_points.map(lambda f:f.buffer(VIIRS_CONFIRM_BUFFER_M))
viirs_buffer_mask=ee.Image(0).byte().paint(viirs_fire_buffer,1).clip(aoi).gt(0).rename('viirs_buffer')
# เป็นความสอดคล้องกับ hotspot เท่านั้น ไม่ยืนยันรายแปลง และใช้จุด calibration ร่วมด้วย
confirmed_burn_mask=burn_mask.And(viirs_buffer_mask).rename('confirmed_burn')
pixel_area=ee.Image.pixelArea()
stats_image=(pixel_area.updateMask(cropland_mask).rename('cropland_area')
    .addBands(pixel_area.updateMask(valid_crop_mask).rename('valid_area'))
    .addBands(pixel_area.updateMask(burn_mask).rename('burn_area'))
    .addBands(pixel_area.updateMask(no_burn_mask).rename('no_burn_area'))
    .addBands(pixel_area.updateMask(confirmed_burn_mask).rename('confirmed_burn_area'))
    .clip(aoi))
assert set(stats_image.bandNames().getInfo())=={'cropland_area','valid_area','burn_area','no_burn_area','confirmed_burn_area'}
print('Burn/No Burn ภายในพื้นที่เกษตรที่ภาพใช้ได้; No Burn = ไม่เข้าเกณฑ์ตรวจพบ')

import pandas as pd
rows=[]
for f in tambons_geojson['features']:
    p=f['properties'];print('สรุปตำบล',p['tambon_name'],flush=True)
    v=stats_image.reduceRegion(reducer=ee.Reducer.sum(),geometry=ee.Geometry(f['geometry']),
        crs=EVAL_CRS,crsTransform=EVAL_TRANSFORM,maxPixels=1e9,tileScale=16).getInfo()
    crop=v.get('cropland_area') or 0; valid=v.get('valid_area') or 0
    rows.append({'tambon_id':str(p['tambon_id']),'tambon_name':p['tambon_name'],
        'burn_pct (%)':100*(v.get('burn_area') or 0)/valid if valid else None,
        'no_burn_pct (%)':100*(v.get('no_burn_area') or 0)/valid if valid else None,
        'valid_observation_pct (%)':100*valid/crop if crop else 0})
df_tambon_overview=pd.DataFrame(rows)
display(df_tambon_overview)

## F · FTW และสถิติรายแปลง — ขั้นนี้ใช้เวลามาก

In [ ]:
import geopandas as gpd
import pandas as pd
import geemap

print("--- เริ่มกระบวนการดึงขอบเขตแปลงจาก Fields of The World (FTW) ---")

# โหลด FTW เพียงครั้งเดียว
import os
# ⚠️ ตั้ง path ของไฟล์ FTW ตามเครื่อง/Colab ของคุณ (หรือกำหนดผ่าน env FTW_PATH)
FTW_FILENAME = "2025_N15E100.parquet"
ftw_path = os.environ.get("FTW_PATH", FTW_FILENAME)
if not os.path.exists(ftw_path):
    try:
        from google.colab import files as _colab_files
        print(f"ไม่พบ {ftw_path} — กรุณาอัปโหลดไฟล์ {FTW_FILENAME}")
        _up = _colab_files.upload()
        ftw_path = FTW_FILENAME if FTW_FILENAME in _up else list(_up.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"ไม่พบไฟล์ FTW: {ftw_path} — วางไฟล์ไว้ข้างโน้ตบุ๊ก หรือตั้ง env FTW_PATH")
ftw_all = gpd.read_parquet(ftw_path)
gdf_tambons = gpd.read_file(geojson_path)

if ftw_all.crs is None:
    raise ValueError("ไฟล์ FTW ไม่มี CRS กำกับ ต้องเช็คไฟล์ต้นทางก่อน")

if ftw_all.crs.to_epsg() != 4326:
    ftw_all = ftw_all.to_crs(epsg=4326)

if gdf_tambons.crs.to_epsg() != 4326:
    gdf_tambons = gdf_tambons.to_crs(epsg=4326)

name_col = "NAME_3" if "NAME_3" in gdf_tambons.columns else "tambon_name"
id_col = "GID_3" if "GID_3" in gdf_tambons.columns else "tambon_id"

# ตัดเหลือเฉพาะกรอบ AOI ก่อน (tile FTW ใหญ่มาก — ลด RAM ก่อนสร้างสำเนา UTM)
_minx, _miny, _maxx, _maxy = gdf_tambons.total_bounds
ftw_all = ftw_all.cx[_minx:_maxx, _miny:_maxy].copy()
print(f"FTW ในกรอบ AOI (bbox): {len(ftw_all):,} แปลง")

# เก็บพื้นที่เดิมของ FTW ก่อน clip เพื่อใช้ตรวจแปลงที่ถูกตัดขอบ
ftw_all = ftw_all.copy()
ftw_all["ftw_origin_id"] = ftw_all.index

ftw_all_utm = ftw_all.to_crs(epsg=32647)
original_area_rai = (
    ftw_all_utm.set_index("ftw_origin_id").geometry.area / 1600
).to_dict()

# Clip FTW กับ AOI เพียงครั้งเดียว ไม่ clip ซ้ำทุกตำบล
ftw_aoi = gpd.clip(ftw_all, gdf_tambons)
ftw_aoi = ftw_aoi.copy()

print(f"FTW หลัง clip กับ AOI: {len(ftw_aoi):,} แปลง")

# ใช้ centroid เพื่อกำหนดว่าแต่ละแปลงอยู่ตำบลใด
# ลดการเรียก gpd.clip ซ้ำทีละตำบล
plot_centroids = ftw_aoi.to_crs(epsg=32647).geometry.centroid.to_crs(epsg=4326)  # คำนวณใน UTM แล้วกลับ WGS84
centroid_gdf = gpd.GeoDataFrame(
    ftw_aoi[["ftw_origin_id"]],
    geometry=plot_centroids,
    crs=ftw_aoi.crs
)

tambon_join = gdf_tambons[[id_col, name_col, "geometry"]].copy()

assigned = gpd.sjoin(
    centroid_gdf,
    tambon_join,
    how="left",
    predicate="within"
)

assigned = assigned[["ftw_origin_id", id_col, name_col]].drop_duplicates(
    subset=["ftw_origin_id"]
)

ftw_aoi = ftw_aoi.merge(
    assigned,
    on="ftw_origin_id",
    how="left"
)

ftw_aoi = ftw_aoi.rename(
    columns={
        id_col: "tambon_id",
        name_col: "tambon_name"
    }
)

# ตัดแปลงที่อยู่นอกตำบลออก
ftw_aoi = ftw_aoi.dropna(
    subset=["tambon_id", "tambon_name"]
).copy()

# คำนวณพื้นที่/centroid ในครั้งเดียว
ftw_utm = ftw_aoi.to_crs(epsg=32647)
ftw_aoi["plot_area_rai"] = ftw_utm.geometry.area / 1600

_cent_wgs = ftw_utm.geometry.centroid.to_crs(epsg=4326)
ftw_aoi["centroid_lon"] = _cent_wgs.x
ftw_aoi["centroid_lat"] = _cent_wgs.y

# ตรวจว่าแปลงถูกตัดจากขอบเขตตำบลหรือไม่
ftw_aoi["original_area_rai"] = ftw_aoi["ftw_origin_id"].map(
    original_area_rai
)

ftw_aoi["is_boundary_fragment"] = (
    ftw_aoi["plot_area_rai"] <
    ftw_aoi["original_area_rai"] * 0.95
)

ftw_aoi["plot_id"] = [
    f"{name}_{i:05d}"
    for name, i in zip(
        ftw_aoi["tambon_name"].astype(str),
        range(len(ftw_aoi))
    )
]

# เก็บเฉพาะคอลัมน์ที่ต้องใช้
cols_to_keep = [
    "plot_id", "tambon_id", "tambon_name", "ftw_origin_id",
    "is_boundary_fragment", "plot_area_rai",
    "centroid_lon", "centroid_lat", "geometry"
]
plots_all = ftw_aoi[cols_to_keep].copy()

print(f"แปลง FTW ที่อยู่ในพื้นที่ศึกษา: {len(plots_all):,}")

# ------------------------------------------------------------
# จุดสำคัญ: ลดจำนวน request ไป Google Earth Engine
# จากเดิม loop แยกทุกตำบล -> batch รวมทั้งพื้นที่
# ------------------------------------------------------------
batch_size = 100   # ⚠️ ลดจาก 5000: กัน error "Request payload size exceeds 10485760 bytes"
from pathlib import Path
import hashlib,json
cache_dir=Path('burnwise_cache');cache_dir.mkdir(exist_ok=True)
cache_config={'pre':[pre_start,pre_end],'post':[post_start,post_end],
 'threshold':float(THRESHOLD),'ndvi':[NDVI_PRE_MIN,NDVI_POST_MAX],
 'grid':[EVAL_CRS,EVAL_TRANSFORM],'worldcover':'ESA/WorldCover/v100',
 'aoi':hashlib.sha256(Path(geojson_path).read_bytes()).hexdigest(),
 'ftw_size':Path(ftw_path).stat().st_size,'ftw_mtime':Path(ftw_path).stat().st_mtime_ns,
 'version':'burnwise_merged_valid_crop_v1'}
cache_key=hashlib.sha256(json.dumps(cache_config,sort_keys=True).encode()).hexdigest()[:20]
cache_path=cache_dir/f'plot_stats_{cache_key}.parquet'
RECOMPUTE_PLOTS=True  # ⚠️ แก้เป็น True เพื่อบังคับคำนวณใหม่ทุกครั้ง (ไม่ใช้ cache)
if cache_path.is_file() and not RECOMPUTE_PLOTS:
    df_stats=pd.read_parquet(cache_path)
    print('ใช้สถิติ GEE ที่บันทึกไว้:',cache_path)
else:
    all_plot_rows = []

    for start in range(0, len(plots_all), batch_size):
        end = min(start + batch_size, len(plots_all))
        batch_gdf = plots_all.iloc[start:end].copy()

        print(f"GEE plot statistics: {start:,} - {end:,} / {len(plots_all):,}")

        batch_send = batch_gdf[["plot_id", "geometry"]].copy()
        ee_fc = geemap.geopandas_to_ee(batch_send)

        stats = stats_image.reduceRegions(
            collection=ee_fc,
            reducer=ee.Reducer.sum(),
            crs=EVAL_CRS,
            crsTransform=EVAL_TRANSFORM,
            tileScale=16
        ).getInfo()

        for f in stats["features"]:
            p = f["properties"]

            cropland_area = p.get("cropland_area") or 0
            burn_area = p.get("burn_area") or 0
            no_burn_area = p.get("no_burn_area") or 0

            valid_area = p.get('valid_area') or 0
            # แปลงที่ไม่มี cropland pixel เลย (WorldCover ไม่ตรงกับ FTW) -> NaN (ไม่ใช่ 0)
            # กันไม่ให้ถูกนับเป็น "No Burn / Green" ทั้งที่ไม่มีข้อมูลตัดสิน
            if valid_area > 0:
                burn_pct = burn_area / valid_area * 100
                no_burn_pct = no_burn_area / valid_area * 100
            else:
                burn_pct = no_burn_pct = float("nan")

            all_plot_rows.append({
                "plot_id": p.get("plot_id"),
                "burn_pct": burn_pct,
                "no_burn_pct": no_burn_pct,
                "cropland_area_m2": cropland_area,
                "valid_area_m2": valid_area,
                "burn_area_m2": burn_area,
                "valid_observation_pct": min(100,100*valid_area/cropland_area) if cropland_area>0 else 0,
            })


    df_stats=pd.DataFrame(all_plot_rows)
    df_stats.to_parquet(cache_path,index=False)
    print('บันทึก checkpoint:',cache_path)
# รวมผลจาก GEE กลับกับ attribute ต้นฉบับ (ส่งไป GEE เฉพาะ plot_id + geometry เพื่อลด payload)

attr_cols = ["plot_id", "tambon_id", "tambon_name", "ftw_origin_id",
             "is_boundary_fragment", "plot_area_rai", "centroid_lon", "centroid_lat"]
df_plots = df_stats.merge(plots_all[attr_cols], on="plot_id", how="left")


df_plots["exclusion_reason"] = None
df_plots.loc[
    df_plots["plot_area_rai"] < 0.1,
    "exclusion_reason"
] = "แปลงเล็กกว่า 0.1 ไร่ (น่าจะเป็นเศษขอบ)"

# ⚠️ แก้ตาม Stage 1 (F): เดิมเหมารวม burn_pct เป็น NaN ทุกกรณีว่า "ไม่มี cropland pixel"
# ทั้งที่จริงมี 2 สาเหตุต่างกันโดยสิ้นเชิง ต้องแยก exclusion_reason ให้ชัดเพื่อแก้ปัญหาได้ถูกจุด:
#   (1) ไม่มี cropland pixel เลย (WorldCover ไม่เห็นเป็นพื้นที่เกษตร) -> ปัญหาเรื่อง land-cover/FTW mismatch
#   (2) มี cropland จริง แต่ไม่มีภาพ Sentinel-2 ใช้ได้เลยในแปลงนี้ -> ปัญหาเรื่องภาพ (เมฆบัง/ไม่มีข้อมูล)
_no_crop_at_all = (df_plots["cropland_area_m2"].fillna(0) == 0) & df_plots["exclusion_reason"].isna()
df_plots.loc[_no_crop_at_all, "exclusion_reason"] = "ไม่มี cropland pixel (ESA WorldCover) ในแปลง"

_crop_no_valid_image = (
    (df_plots["cropland_area_m2"].fillna(0) > 0)
    & (df_plots["valid_area_m2"].fillna(0) == 0)
    & df_plots["exclusion_reason"].isna()
)
df_plots.loc[_crop_no_valid_image, "exclusion_reason"] = (
    "มี cropland แต่ไม่มีภาพ Sentinel-2 ที่ใช้ได้เลยในแปลงนี้ (เมฆบัง/ไม่มีข้อมูลทั้งสองช่วง)"
)

print(f"แปลงที่ไม่มี cropland pixel เลย: {int(_no_crop_at_all.sum()):,}")
print(f"แปลงที่มี cropland แต่ไม่มีภาพใช้ได้เลย: {int(_crop_no_valid_image.sum()):,}")
print(f"รวมแปลงที่ burn_pct เป็น NaN: {int(df_plots['burn_pct'].isna().sum()):,}")

df_plots.loc[(df_plots['valid_observation_pct'] < MIN_VALID_OBSERVATION_PCT)
    & df_plots['exclusion_reason'].isna(),'exclusion_reason']='ภาพใช้ได้ครอบคลุมพื้นที่เกษตรต่ำกว่า 80%'
df_plots['unknown_pct']=100-df_plots['valid_observation_pct']
assert df_plots['plot_id'].is_unique
assert ((df_plots['burn_pct'].dropna()>=0)&(df_plots['burn_pct'].dropna()<=100.001)).all()
df_plots["classification_pct"] = (
    df_plots["burn_pct"] + df_plots["no_burn_pct"]
)

print("--------------------------------------------------")
print(f"แปลง FTW ทั้งหมดในพื้นที่ศึกษา: {len(df_plots):,}")
print(f'ค่าเฉลี่ย burn_pct: {df_plots["burn_pct"].mean():.2f}%')
print(f'ค่าเฉลี่ย no_burn_pct: {df_plots["no_burn_pct"].mean():.2f}%')
print("--------------------------------------------------")
print("Binary classification พร้อมใช้งาน: Burn / No Burn")

df_plots.head()


## G · สถานะรายแปลงและ Tier

In [ ]:
BURN_PLOT_THRESHOLD=10.0

def burn_status(row):
    if pd.isna(row['burn_pct']) or pd.notna(row['exclusion_reason']):
        return 'ข้อมูลไม่เพียงพอ'
    return 'Burn' if row['burn_pct'] >= BURN_PLOT_THRESHOLD else 'No Burn'
df_plots['burn_status']=df_plots.apply(burn_status,axis=1)
print(df_plots['burn_status'].value_counts())
print('burn_pct หารด้วยพื้นที่เกษตรที่มีภาพใช้ได้; No Burn = ไม่เข้าเกณฑ์ตรวจพบ ไม่ใช่ยืนยันว่าไม่เผา')

# ============================================================
# I.2 Tier รายแปลง: Green / Yellow / Red
# ============================================================
TIER_YELLOW_MIN = BURN_PLOT_THRESHOLD   # 10% — ตรงกับเกณฑ์ Burn/No Burn
TIER_RED_MIN = 50.0

def burn_tier(row):
    b = row['burn_pct']
    if pd.isna(b) or pd.notna(row.get('exclusion_reason')):
        return 'Unknown'
    if b >= TIER_RED_MIN:
        return 'Red'
    if b >= TIER_YELLOW_MIN:
        return 'Yellow'
    return 'Green'

TIER_LABEL = {
    'Green':   '🟢 Green — No Burn',
    'Yellow':  '🟡 Yellow — Burn บางส่วน (ตรวจสอบ/ติดตาม)',
    'Red':     '🔴 Red — สัญญาณ Burn อย่างน้อย 50% ของพื้นที่ประเมิน (เร่งด่วน)',
    'Unknown': '⚪ Unknown — ข้อมูลไม่เพียงพอ',
}
TIER_COLOR = {'Green': '#43A047', 'Yellow': '#FBC02D', 'Red': '#E53935', 'Unknown': '#9E9E9E'}

df_plots['burn_tier'] = df_plots.apply(burn_tier, axis=1)
df_plots['burn_tier_label'] = df_plots['burn_tier'].map(TIER_LABEL)

# ความสอดคล้อง: Tier Yellow/Red ต้องเป็น Burn เสมอ (เช็คกันเกณฑ์ไม่ตรงกัน)
_bad = df_plots[(df_plots['burn_tier'].isin(['Yellow', 'Red'])) & (df_plots['burn_status'] != 'Burn')]
assert len(_bad) == 0, f'Tier กับ burn_status ไม่สอดคล้อง {len(_bad)} แปลง'

tier_summary = (
    df_plots.groupby('burn_tier')
    .agg(plots=('plot_id', 'count'), area_rai=('plot_area_rai', 'sum'))
    .reindex(['Green', 'Yellow', 'Red', 'Unknown']).fillna(0)
)
tier_summary['plots_pct'] = (tier_summary['plots'] / tier_summary['plots'].sum() * 100).round(1)
tier_summary['area_rai'] = tier_summary['area_rai'].round(1)
print(f'เกณฑ์ Tier: Yellow >= {TIER_YELLOW_MIN}%, Red >= {TIER_RED_MIN}%')
tier_summary

## H · ถนน, รายได้ตำบล และ Access Gap

In [ ]:
collection_points = [
    {"name": "บริษัท รวมผลใบโอเพาเวอร์ จำกัด", "type": "โรงไฟฟ้าชีวมวล", "lat": 15.7196, "lon": 100.0872},
    {"name": "บริษัท เกษตรไทย อินเตอร์เนชั่นแนล ชูการ์ คอร์ปอเรชั่น จำกัด (มหาชน) สาขา 3 (KTIS 3)", "type": "โรงงานน้ำตาล + โรงไฟฟ้าชีวมวลในเครือ", "lat": 15.3773, "lon": 100.2419},
    {"name": "ไทยส่งเสริม โรงสีไฟ", "type": "โรงสีข้าว (แหล่งชีวมวล-แกลบ)", "lat": 15.6357, "lon": 100.4804},
    {"name": "บริษัท รวมผลอุตสาหกรรมนครสวรรค์ จำกัด (โรงงานน้ำตาลรวมผล)", "type": "โรงงานน้ำตาล", "lat": 15.7917, "lon": 100.0863},
    {"name": "บริษัท โรงสีไพเกียรติรุ่งเรือง จำกัด", "type": "โรงสีข้าว (แหล่งชีวมวล-แกลบ)", "lat": 15.6533, "lon": 100.5698},
    {"name": "โรงไฟฟ้า TRUE ENERGY CO.,LTD.(นครสวรรค์)", "type": "โรงไฟฟ้าชีวมวล", "lat": 15.8617, "lon": 100.5842},
    {"name": "บริษัท เกษตรไทยไบโอเพาเวอร์ จำกัด", "type": "โรงไฟฟ้าชีวมวล", "lat": 15.2865, "lon": 100.419},
    {"name": "ลานรับซื้อผลผลิตการเกษตร ดอนคา", "type": "จุดรับซื้อผลผลิตเกษตร", "lat": 15.6512, "lon": 100.558},
    {"name": "บริษัท เกษตรไทย ไบโอเอทานอล จำกัด (KTBE)", "type": "โรงงานแปรรูปชีวมวล/เอทานอล", "lat": 15.2842, "lon": 100.4173},
    {"name": "บริษัท นครสวรรค์ ไบโอแก๊ส จำกัด", "type": "โรงผลิตก๊าซชีวภาพ/พลังงาน", "lat": 15.7485, "lon": 100.321},
]
print(f'จำนวนจุดรับซื้อ/โรงงานทั้งหมด: {len(collection_points)}')


import osmnx as ox
import networkx as nx
G=ox.load_graphml('burnwise_roads.graphml')
print(f'ถนนพร้อมใช้: {len(G.nodes):,} nodes, {len(G.edges):,} edges')
print('ระยะทางเป็นเส้นทางจาก node ที่ใกล้ centroid ไม่รวมทางจากแปลงเข้าสู่ถนน; จุดรับซื้อต้องตรวจสอบบริการจริง')

plot_lons = df_plots['centroid_lon'].tolist()
plot_lats = df_plots['centroid_lat'].tolist()
plot_nodes = ox.distance.nearest_nodes(G, X=plot_lons, Y=plot_lats)
df_plots['road_node'] = plot_nodes

collection_nodes = ox.distance.nearest_nodes(
    G, X=[p['lon'] for p in collection_points], Y=[p['lat'] for p in collection_points])

# กราฟย้อนทิศ + hub เสมือน: ระยะทางจาก hub ไปยัง node ใดๆ บนกราฟย้อนทิศ
# = ระยะทางจาก node นั้นไปยังจุดรับซื้อที่ใกล้ที่สุดบนกราฟจริง (สำคัญ เพราะถนนมีทิศทางเดินรถ)
G_rev = G.reverse(copy=True)
HUB = '__collection_hub__'
G_rev.add_node(HUB)
for cn in collection_nodes:
    G_rev.add_edge(HUB, cn, length=0)

dist_to_nearest_collection_m = nx.single_source_dijkstra_path_length(G_rev, HUB, weight='length')

df_plots['distance_to_collection_km'] = df_plots['road_node'].map(
    lambda n: round(dist_to_nearest_collection_m[n] / 1000, 2) if n in dist_to_nearest_collection_m else None
)

n_unreachable = df_plots['distance_to_collection_km'].isna().sum()
print(f'คำนวณระยะทางถนนสำเร็จ — แปลงที่หาเส้นทางไม่ได้: {n_unreachable} จาก {len(df_plots)}')
if n_unreachable > 0:
    print('  (ถ้ามีจำนวนมาก ลองเพิ่ม buffer_deg ในเซลล์ดึงถนนด้านบนแล้วรันใหม่)')


import pandas as pd
socio=pd.read_csv('thatako_socioeconomic_data_2568.csv',encoding='utf-8-sig',dtype={'รหัสพื้นที่':str})
socio['tambon_id']=socio['รหัสพื้นที่'].str.replace('-','',regex=False)
socio=socio.rename(columns={
    'รายได้เฉลี่ยต่อครัวเรือน (บาท/ปี)':'tambon_income_baht_year',
    'รายจ่ายเฉลี่ยต่อครัวเรือน (บาท/ปี)':'tambon_expense_baht_year',
    'อัตราครัวเรือนไม่มีหนี้ (%)':'tambon_debt_free_pct'})
socio_cols=['tambon_income_baht_year','tambon_expense_baht_year','tambon_debt_free_pct']
df_plots['tambon_id']=df_plots['tambon_id'].astype(str).str.replace(r'\.0$','',regex=True)
df_plots=df_plots.drop(columns=socio_cols,errors='ignore').merge(
    socio[['tambon_id']+socio_cols],on='tambon_id',how='left',validate='many_to_one')
assert df_plots['tambon_income_baht_year'].notna().all(),'รายได้เชื่อมไม่ครบ ตรวจรหัส DOPA'
print('เชื่อมรายได้ครบ',len(df_plots),'แปลง; เป็นค่าเฉลี่ยตำบล ไม่ใช่รายได้เจ้าของแปลง')

def minmax_norm(series):
    x=pd.to_numeric(series,errors='coerce'); span=x.max()-x.min()
    if pd.isna(span): return x*float('nan')
    if span==0:return x.where(x.isna(),0.5)
    return (x-x.min())/span
df_plots['dist_norm']=minmax_norm(df_plots['distance_to_collection_km'])
df_plots['plot_norm']=1-minmax_norm(df_plots['plot_area_rai'])
df_plots['income_gap_norm']=1-minmax_norm(df_plots['tambon_income_baht_year'])
df_plots['access_gap_index_no_viirs']=0.6*df_plots['dist_norm']+0.4*df_plots['plot_norm']
df_plots['access_gap_index']=0.5*df_plots['dist_norm']+0.3*df_plots['plot_norm']+0.2*df_plots['income_gap_norm']
print('น้ำหนักทดลอง 50/30/20; Access Gap ไม่ใช้กำหนดว่าเผาหรือไม่เผา')

## I · ตรวจสอบซ้ำและตารางหลัก

In [ ]:
# ============================================================
# สรุป Binary status ระดับแปลง
# ============================================================

# ใช้ Access Gap ต่อได้หากต้องการวิเคราะห์เชิงพื้นที่ แต่ไม่ใช้เพื่อกำหนด Burn/No Burn

# ⚠️ แก้ตาม Stage 1 (I): เดิมตรวจสอบซ้ำเฉพาะกรณี "ใกล้ threshold" อย่างเดียว
# ทั้งที่มีอีกหลายเหตุผลที่ผลลัพธ์แปลงนั้นไม่น่าเชื่อถือพอจะใช้ฟันธงได้ — รวมทุกเหตุผลที่เข้าเกณฑ์ไว้ในแปลงเดียวกันได้
def _verification_reasons(r):
    reasons = []
    if pd.notna(r.get('burn_pct')) and abs(r['burn_pct'] - BURN_PLOT_THRESHOLD) <= 3:
        reasons.append('สัดส่วน Burn ใกล้เกณฑ์ตัดสิน (±3 จุด)')
    if r.get('burn_tier') == 'Unknown':
        reasons.append('Tier อ่านไม่ได้ (Unknown) — ข้อมูลไม่เพียงพอ')
    if r.get('is_boundary_fragment') is True:
        reasons.append('ขอบแปลงถูกตัดจากขอบเขตตำบล (fragment) — พื้นที่อาจไม่ตรงแปลงจริง')
    if pd.isna(r.get('distance_to_collection_km')):
        reasons.append('หาเส้นทางถนนไปจุดรับซื้อที่ใกล้ที่สุดไม่เจอ')
    if not reasons:
        return None
    return 'ตรวจสอบซ้ำ: ' + '; '.join(reasons)

df_plots['needs_verification'] = df_plots.apply(_verification_reasons, axis=1)
print('จำนวนแปลงที่ถูกตั้งค่าตรวจสอบซ้ำ (รวมทุกเหตุผล):', df_plots['needs_verification'].notna().sum())

print('จำนวน Burn:', (df_plots['burn_status'] == 'Burn').sum())
print('จำนวน No Burn:', (df_plots['burn_status'] == 'No Burn').sum())
print('จำนวนที่ควรตรวจสอบซ้ำ:', df_plots['needs_verification'].notna().sum())


master_cols = [
    'plot_id', 'tambon_id', 'tambon_name', 'plot_area_rai',
    'burn_pct', 'no_burn_pct', 'burn_status', 'burn_tier', 'burn_tier_label',
    'classification_pct', 'valid_observation_pct', 'unknown_pct',
    'cropland_area_m2', 'valid_area_m2', 'burn_area_m2',
    'is_boundary_fragment', 'exclusion_reason',
    'centroid_lat', 'centroid_lon',
    'distance_to_collection_km', 'access_gap_index', 'access_gap_index_no_viirs',
    'tambon_income_baht_year', 'tambon_expense_baht_year', 'tambon_debt_free_pct',
    'needs_verification'
]

# เก็บเฉพาะคอลัมน์ที่มีอยู่จริง เพื่อให้ notebook ยังทำงานได้แม้ไม่ได้รันส่วน Access Gap
master_cols = [c for c in master_cols if c in df_plots.columns]
master_plots = df_plots[master_cols].copy()

print(f'จำนวนแปลงทั้งหมด: {len(master_plots)}')
print('Burn:', (master_plots['burn_status'] == 'Burn').sum())
print('No Burn:', (master_plots['burn_status'] == 'No Burn').sum())

master_plots.head(10)

# ============================================================
# M.1 สร้างผลระดับตำบลสำหรับกำหนดมาตรการ
# ============================================================

# ใช้ค่า burn_pct จาก Sentinel-2 และ access_gap_index เฉลี่ยของแปลงในแต่ละตำบล
_eligible=master_plots[master_plots['burn_status']!='ข้อมูลไม่เพียงพอ'].copy()
_sums=_eligible.groupby(['tambon_id','tambon_name']).agg(
    burn_area_m2=('burn_area_m2','sum'),valid_area_m2=('valid_area_m2','sum'),
    access_gap_index=('access_gap_index','mean'))
_all=master_plots.groupby(['tambon_id','tambon_name']).agg(plot_count=('plot_id','count'),
    all_crop_m2=('cropland_area_m2','sum'))
tambon_measure=_all.join(_sums).reset_index()
tambon_measure['burn_pct']=100*tambon_measure['burn_area_m2']/tambon_measure['valid_area_m2']
tambon_measure['no_burn_pct']=100-tambon_measure['burn_pct']
tambon_measure['evaluated_crop_pct']=100*tambon_measure['valid_area_m2']/tambon_measure['all_crop_m2'].replace(0,float('nan'))

def tambon_burn_status(row):
    if pd.isna(row['burn_pct']) or row['evaluated_crop_pct'] < MIN_VALID_OBSERVATION_PCT:
        return 'ข้อมูลไม่เพียงพอ'
    if row['burn_pct'] >= BURN_PLOT_THRESHOLD:
        return 'Burn'
    return 'No Burn'

tambon_measure['burn_status'] = tambon_measure.apply(tambon_burn_status, axis=1)

def assign_measure(row):
    if row['burn_status'] == 'ข้อมูลไม่เพียงพอ' or pd.isna(row['access_gap_index']):
        return 'รอข้อมูลเพิ่มเติม — ยังไม่กำหนดมาตรการ'
    if row['burn_status'] == 'No Burn':
        return 'ติดตามพื้นที่ที่ไม่เข้าเกณฑ์ตรวจพบ + ตรวจหลักฐานการจัดการแปลง'
    if row['access_gap_index'] >= 0.60:
        return 'ช่วยเหลือด้านโลจิสติกส์/เครื่องจักร + ทางเลือกจัดการเศษวัสดุ'
    return 'ลดการเผา + สื่อสาร/ติดตาม + ตรวจสอบซ้ำพื้นที่เสี่ยง'

tambon_measure['measure'] = tambon_measure.apply(assign_measure, axis=1)
tambon_measure['measure_group'] = tambon_measure.apply(
    lambda r: (
        'ข้อมูลไม่เพียงพอ' if 'รอข้อมูล' in r['measure'] else
        'No Burn' if r['burn_status'] == 'No Burn' else
        'Burn + Access Gap สูง' if r['access_gap_index'] >= 0.60 else
        'Burn + Access Gap ต่ำ'
    ), axis=1
)

print('สรุปมาตรการรายตำบล')
print(tambon_measure[
    ['tambon_name', 'burn_pct', 'access_gap_index', 'burn_status', 'measure']
].to_string(index=False))

# ============================================================
# M.4 Save ผลระดับตำบลและระดับแปลง
# ============================================================

tambon_result = tambon_measure[
    ['tambon_id', 'tambon_name', 'burn_pct', 'no_burn_pct',
     'access_gap_index', 'plot_count', 'burn_status',
     'measure_group', 'measure']
].copy()

tambon_result = tambon_result.sort_values(
    ['measure_group', 'tambon_name']
).reset_index(drop=True)

# ---------- ระดับแปลง ----------
master_plots['measure_group'] = master_plots.apply(
    lambda r: (
        'ข้อมูลไม่เพียงพอ' if (pd.isna(r['burn_status']) or r['burn_status'] == 'ข้อมูลไม่เพียงพอ' or pd.isna(r.get('access_gap_index'))) else
        'No Burn' if r['burn_status'] == 'No Burn' else
        ('Burn + Access Gap สูง'
         if pd.notna(r.get('access_gap_index')) and r['access_gap_index'] >= 0.60
         else 'Burn + Access Gap ต่ำ')
    ),
    axis=1
)

master_plots['measure'] = master_plots['measure_group'].map({
    'No Burn': 'ติดตามพื้นที่ที่ไม่เข้าเกณฑ์ตรวจพบ + ตรวจหลักฐานการจัดการแปลง',
    'Burn + Access Gap สูง': 'ช่วยเหลือด้านโลจิสติกส์/เครื่องจักร + ทางเลือกจัดการเศษวัสดุ',
    'Burn + Access Gap ต่ำ': 'ลดการเผา + สื่อสาร/ติดตาม + ตรวจสอบซ้ำพื้นที่เสี่ยง',
    'ข้อมูลไม่เพียงพอ': 'รอข้อมูลเพิ่มเติม — ยังไม่กำหนดมาตรการ'
})

print('ตารางมาตรการเบื้องต้นพร้อม — ต้องตรวจสอบก่อนใช้งานจริง')


## J · Panel และส่งออก CSV

In [ ]:
status_panel = master_plots.groupby(['tambon_name', 'burn_status']).size().unstack(fill_value=0)
for col in ['Burn', 'No Burn', 'ข้อมูลไม่เพียงพอ']:
    if col not in status_panel.columns:
        status_panel[col] = 0
status_panel = status_panel[['Burn', 'No Burn', 'ข้อมูลไม่เพียงพอ']]
status_panel['รวมทั้งหมด'] = status_panel.sum(axis=1)
status_panel


# ============================================================
# N) Export ผลลัพธ์ทั้งหมด — ทำครั้งเดียว
# ============================================================

import os

export_files = []

master_plots.to_csv(
    "burnwise_master_plots.csv",
    index=False,
    encoding="utf-8-sig"
)
export_files.append("burnwise_master_plots_last1.csv")

status_panel.to_csv(
    "burnwise_burn_no_burn_panel.csv",
    encoding="utf-8-sig"
)
export_files.append("burnwise_burn_no_burn_panel_last1.csv")

df_tambon_overview.to_csv(
    "burnwise_tambon_overview.csv",
    index=False,
    encoding="utf-8-sig"
)
export_files.append("burnwise_tambon_overview_last1.csv")

if "tambon_result" in globals():
    tambon_result.to_csv(
        "burnwise_tambon_measures_last1.csv",
        index=False,
        encoding="utf-8-sig"
    )
    export_files.append("burnwise_tambon_measures_last1.csv")

if "gdf_measure" in globals():
    gdf_measure.to_crs(epsg=4326).to_file(
        "burnwise_tambon_measures.geojson",
        driver="GeoJSON"
    )
    export_files.append("burnwise_tambon_measures.geojson")

if "master_plots" in globals() and "measure" in master_plots.columns:
    master_plots.to_csv(
        "burnwise_master_plots_with_measures_last1.csv",
        index=False,
        encoding="utf-8-sig"
    )
    export_files.append("burnwise_master_plots_with_measures_last1.csv")

if "measure_map" in globals():
    try:
        measure_map.to_html("burnwise_measure_map.html")
        export_files.append("burnwise_measure_map.html")
    except Exception as e:
        print("ไม่สามารถบันทึก HTML map ได้:", e)

print("========================================")
print("บันทึกผลเรียบร้อย")
print("========================================")

for f in export_files:
    print(" -", f)

try:
    from google.colab import files as colab_files
    print("\nถ้าต้องการดาวน์โหลด ให้ใช้:")
    print("files.download('ชื่อไฟล์')")
except ImportError:
    print("\nNotebook นี้ไม่ได้รันบน Google Colab")
    print("ไฟล์ถูกบันทึกในโฟลเดอร์ปัจจุบัน")

score_panel=master_plots.groupby(['tambon_name','burn_tier']).size().unstack(fill_value=0).reindex(columns=['Green','Yellow','Red','Unknown'],fill_value=0)
score_panel['รวมทั้งหมด']=score_panel.sum(axis=1)
assert score_panel['รวมทั้งหมด'].sum()==len(master_plots)
score_panel.to_csv('burnwise_score_panel_last1.csv',encoding='utf-8-sig')
print('เพิ่ม burnwise_score_panel_last1.csv (Green/Yellow/Red/Unknown)')


## K · แสดงกราฟผลหลัก (Code cell 11)

รันหลัง J หรืออ่าน CSV ของ J ถ้ามีอยู่แล้วได้เลย ไม่ต้องรอ GISTDA

กราฟ: สถานะรายตำบล, สัดส่วนพื้นที่สัญญาณเผา, coverage รายแปลง, เหตุผล Unknown, สัญญาณเผาเทียบ Access Gap, ระยะทางถนน

แกนพื้นที่เผารายตำบลใช้ภาพรวม raster จาก E; สถานะรายตำบลใช้จำนวนแปลงจากตารางหลัก ฐานทั้งสองต่างกัน

ติดตั้ง Plotly ถ้ายังไม่มีด้วย `%pip install plotly` แล้วรันเซลล์เดียวด้านล่าง เมื่อเปลี่ยนเกณฑ์/ข้อมูลให้รัน upstream ตามเดิมก่อนแล้วรัน K ใหม่

**นำไปใช้ใน Streamlit:** อ่าน JSON ด้วย `plotly.io.from_json(Path('...json').read_text(encoding='utf-8'))` แล้วเรียก `st.plotly_chart(fig)` หรือเรียก `make_burnwise_charts()` ด้วยตารางหลังใช้ตัวกรองของเว็บ เพื่อให้กราฟเปลี่ยนตามตัวกรอง ต้องนำฟังก์ชันและค่าคงที่จาก K ไปไว้ในแอปก่อน


In [ ]:
%pip install --upgrade "nbformat>=4.2.0"

In [ ]:
# K · กราฟสำหรับตรวจผลและนำไปใช้ใน dashboard
# รันหลัง J; หรืออ่าน CSV ที่ส่งออกไว้ได้โดยไม่ต้องดึง GEE ใหม่
from pathlib import Path
import json
import numpy as np
import pandas as pd
try:
    import plotly.express as px
    import plotly.graph_objects as go
    import plotly.io as pio
except ImportError as exc:
    raise ImportError('ติดตั้ง Plotly ก่อน: %pip install plotly แล้วรันเซลล์นี้ใหม่') from exc
from IPython.display import display, HTML

BURNWISE_COLORS = {'Burn': '#C62E33', 'No Burn': '#EF934C', 'ข้อมูลไม่เพียงพอ': '#B9C2CC'}
STATUS_ORDER = list(BURNWISE_COLORS)
STATUS_LABELS = {'Burn': 'ตรวจพบสัญญาณเผา', 'No Burn': 'ไม่เข้าเกณฑ์ตรวจพบ',
                 'ข้อมูลไม่เพียงพอ': 'ข้อมูลไม่เพียงพอ'}


def make_burnwise_charts(plots, overview=None, burn_threshold=10.0, coverage_threshold=80.0):
    """สร้าง Plotly figures จากตารางจริง ไม่แก้ค่าหรือจัดสถานะใหม่ให้แปลง."""
    required = {'plot_id', 'tambon_name', 'burn_status', 'burn_pct',
                'valid_observation_pct', 'exclusion_reason'}
    missing = required - set(plots.columns)
    if missing:
        raise ValueError(f'ตารางยังไม่ใช่ schema ใหม่ ขาดคอลัมน์: {sorted(missing)}; รัน G–J ก่อน')
    if plots.empty:
        raise ValueError('ตารางแปลงว่าง — ตรวจ F–J ก่อนสร้างกราฟ')
    d = plots.copy()
    if d['plot_id'].isna().any() or d['plot_id'].duplicated().any():
        raise ValueError('plot_id ว่างหรือซ้ำ ต้องแก้ตารางหลักก่อนแสดงจำนวนแปลง')
    if d['tambon_name'].isna().any() or not d['burn_status'].isin(STATUS_ORDER).all():
        raise ValueError('มีชื่อตำบลว่าง/สถานะไม่ตรง schema ใหม่ ต้องตรวจ G–J')
    for col in ['burn_pct', 'valid_observation_pct', 'distance_to_collection_km', 'access_gap_index']:
        if col in d:
            original = d[col]
            numeric = pd.to_numeric(original, errors='coerce')
            if (original.notna() & numeric.isna()).any():
                raise ValueError(f'{col} มีค่าที่แปลงเป็นตัวเลขไม่ได้')
            if np.isinf(numeric.dropna()).any():
                raise ValueError(f'{col} มี infinity ต้องแก้ข้อมูลต้นทาง')
            d[col] = numeric
    for col in ['burn_pct', 'valid_observation_pct']:
        if not d[col].dropna().between(0, 100.001).all():
            raise ValueError(f'{col} อยู่นอกช่วง 0–100')
    if 'access_gap_index' in d and not d['access_gap_index'].dropna().between(0, 1).all():
        raise ValueError('Access Gap อยู่นอกช่วง 0–1')
    if 'distance_to_collection_km' in d and (d['distance_to_collection_km'].dropna() < 0).any():
        raise ValueError('ระยะทางติดลบ ต้องตรวจ H')

    figures, tables, notes = {}, {}, []
    counts = (d.groupby(['tambon_name', 'burn_status']).size().unstack(fill_value=0)
              .reindex(columns=STATUS_ORDER, fill_value=0).sort_index())
    assert int(counts.to_numpy().sum()) == len(d)
    totals = counts.sum(axis=1)
    percentages = counts.div(totals, axis=0) * 100
    fig = go.Figure()
    for status in STATUS_ORDER:
        custom = np.column_stack([counts[status].to_numpy(), totals.to_numpy()])
        fig.add_bar(name=STATUS_LABELS[status], x=counts.index, y=percentages[status],
                    marker_color=BURNWISE_COLORS[status], customdata=custom,
                    hovertemplate='%{x}<br>%{fullData.name}: %{y:.1f}%<br>'
                                  'จำนวน %{customdata[0]:,.0f} / %{customdata[1]:,.0f} แปลง<extra></extra>')
    fig.update_layout(barmode='stack', title='สถานะรายตำบล — รวมแปลงข้อมูลไม่เพียงพอ',
                      xaxis_title='ตำบล', yaxis_title='สัดส่วนแปลงทั้งหมดในตำบล (%)')
    fig.update_yaxes(range=[0, 100])
    figures['01_tambon_status'] = fig
    tables['tambon_status_counts'] = counts.reset_index()
    tables['tambon_status_percent'] = percentages.reset_index()

    # ฐานพื้นที่ของ E = raster เกษตรใน AOI ไม่ใช่ค่าเฉลี่ยเปอร์เซ็นต์รายแปลง
    overview_cols = ['tambon_name', 'burn_pct (%)', 'valid_observation_pct (%)']
    if overview is not None and not overview.empty:
        missing_overview = set(overview_cols) - set(overview.columns)
        if missing_overview:
            raise ValueError(f'ตารางภาพรวมขาด {sorted(missing_overview)}; รัน E/J ให้ครบ')
        o = overview[overview_cols].copy()
        if o['tambon_name'].isna().any() or o['tambon_name'].duplicated().any():
            raise ValueError('ชื่อตำบลในภาพรวมว่างหรือซ้ำ')
        for col in overview_cols[1:]:
            src = o[col]; o[col] = pd.to_numeric(src, errors='coerce')
            if (src.notna() & o[col].isna()).any() or not o[col].dropna().between(0, 100.001).all():
                raise ValueError(f'ค่า {col} ในภาพรวมผิดรูปแบบ/ช่วง')
        usable = o.dropna(subset=overview_cols[1:]).sort_values('burn_pct (%)', ascending=False)
        o['คุณภาพภาพ'] = np.where(o['valid_observation_pct (%)'].fillna(-1) >= coverage_threshold,
                                     'ผ่านเกณฑ์ coverage', 'coverage ต่ำ/ไม่ทราบ')
        usable = usable.merge(o[['tambon_name', 'คุณภาพภาพ']], on='tambon_name', validate='one_to_one')
        if not usable.empty:
            figures['02_tambon_burn_area'] = px.bar(usable, x='tambon_name', y='burn_pct (%)',
                color='คุณภาพภาพ', color_discrete_map={'ผ่านเกณฑ์ coverage': '#C62E33',
                'coverage ต่ำ/ไม่ทราบ': '#B9C2CC'}, custom_data=['valid_observation_pct (%)'],
                title='พื้นที่ตรวจพบสัญญาณเผา — ฐานพื้นที่เกษตรที่ภาพใช้ได้ใน AOI')
            figures['02_tambon_burn_area'].update_traces(hovertemplate='%{x}<br>สัญญาณเผา %{y:.1f}%<br>'
                'coverage %{customdata[0]:.1f}%<extra>%{fullData.name}</extra>')
            figures['02_tambon_burn_area'].update_yaxes(title='สัดส่วนพื้นที่เกษตรที่มีภาพใช้ได้ (%)', range=[0,100])
        tables['tambon_raster_overview'] = o
        notes.append(f'ภาพรวมตำบลไม่มีค่าครบ {len(o)-len(usable):,} ตำบล; '
                     'แท่งสีเทามี coverage ต่ำ จึงยังไม่ควรใช้จัดอันดับเพื่อมาตรการ')
    else:
        notes.append('ไม่มีตารางภาพรวม E: ข้ามกราฟพื้นที่เผารายตำบล ไม่สร้างตัวเลขแทน')

    figures['03_observation_coverage'] = px.histogram(d.dropna(subset=['valid_observation_pct']),
        x='valid_observation_pct', color='burn_status', nbins=20,
        category_orders={'burn_status': STATUS_ORDER}, color_discrete_map=BURNWISE_COLORS,
        barmode='stack', title='ความครอบคลุมภาพรายแปลง — รวมทุกสถานะ',
        labels={'valid_observation_pct': 'พื้นที่เกษตรที่มีภาพใช้ได้ (%)', 'burn_status': 'สถานะ'})
    figures['03_observation_coverage'].add_vline(x=coverage_threshold, line_dash='dash',
        line_color='#5D6773', annotation_text=f'เกณฑ์ทดลอง {coverage_threshold:g}%')
    figures['03_observation_coverage'].update_yaxes(title='จำนวนแปลง')
    figures['03_observation_coverage'].update_xaxes(range=[0,100])
    notes.append(f'coverage ไม่มีค่า {d["valid_observation_pct"].isna().sum():,} แปลง; ไม่แสดงใน histogram')

    unknown = d[d['burn_status'].eq('ข้อมูลไม่เพียงพอ')].copy()
    reasons = (unknown['exclusion_reason'].fillna('ยังไม่ระบุเหตุผล')
               .replace('', 'ยังไม่ระบุเหตุผล').value_counts()
               .rename_axis('เหตุผล').reset_index(name='จำนวนแปลง'))
    tables['unknown_reasons'] = reasons
    if not reasons.empty:
        figures['04_unknown_reasons'] = px.bar(reasons.sort_values('จำนวนแปลง'),
            x='จำนวนแปลง', y='เหตุผล', orientation='h', text='จำนวนแปลง',
            title='เหตุผลที่แปลงยังประเมินไม่ได้ — ตามป้ายในตารางหลัก')
        figures['04_unknown_reasons'].update_traces(marker_color='#B9C2CC', textposition='outside')
    else:
        notes.append('ไม่มีแปลงข้อมูลไม่เพียงพอ จึงข้ามกราฟเหตุผล')

    if 'access_gap_index' in d:
        eligible = d[d['burn_status'].isin(['Burn', 'No Burn'])].copy()
        assess = eligible.dropna(subset=['burn_pct', 'access_gap_index'])
        # แบ่งสัดส่วนตามสถานะก่อนสุ่ม ให้กราฟไม่หนักเกินไป; จำนวนจริงใช้กราฟสถานะด้านบน
        cap = 6000
        if len(assess) > cap:
            groups = list(assess.groupby('burn_status', sort=True))
            budgets = [int(cap * len(g) / len(assess)) for _, g in groups]
            for j in range(cap - sum(budgets)):
                budgets[j % len(groups)] += 1
            sample = pd.concat([g.sample(n=n, random_state=42) for (_, g), n in zip(groups, budgets)])
        else:
            sample = assess
        if not sample.empty:
            figures['05_burn_access_gap'] = px.scatter(sample, x='access_gap_index', y='burn_pct',
                color='burn_status', color_discrete_map=BURNWISE_COLORS, opacity=0.45,
                category_orders={'burn_status': STATUS_ORDER}, hover_name='plot_id',
                hover_data=['tambon_name', 'valid_observation_pct'], render_mode='webgl',
                title=f'สัญญาณเผาเทียบ Access Gap — แสดง {len(sample):,}/{len(assess):,} แปลงที่มีข้อมูล',
                labels={'access_gap_index': 'Access Gap (0–1; สูง = ข้อจำกัดมาก)',
                        'burn_pct': 'สัดส่วนพื้นที่สัญญาณเผา (%)', 'burn_status': 'สถานะ'})
            figures['05_burn_access_gap'].update_traces(marker_size=5)
            figures['05_burn_access_gap'].add_vline(x=0.60, line_dash='dash', line_color='#EF934C')
            figures['05_burn_access_gap'].add_hline(y=burn_threshold, line_dash='dash', line_color='#C62E33')
            figures['05_burn_access_gap'].update_xaxes(range=[0,1])
            figures['05_burn_access_gap'].update_yaxes(range=[0,100])
        tables['access_scatter_sample'] = sample
        notes.append(f'กราฟกระจายไม่รวม Unknown {len(unknown):,} แปลง และแปลงข้อมูลแกนไม่ครบ '
                     f'{len(eligible)-len(assess):,} แปลง; เป็นความสัมพันธ์เชิงสำรวจ ไม่ใช่เหตุและผล')
    else:
        notes.append('ไม่มี Access Gap: ข้ามกราฟกระจาย ให้ตรวจ H')

    if 'distance_to_collection_km' in d:
        road = d.dropna(subset=['distance_to_collection_km'])
        if not road.empty:
            figures['06_road_distance'] = px.histogram(road, x='distance_to_collection_km', nbins=35,
                title='การกระจายระยะทางตามโครงข่ายถนนถึงจุดรับซื้อ',
                labels={'distance_to_collection_km': 'ระยะทางตามกราฟถนน (กม.)'})
            figures['06_road_distance'].update_traces(marker_color='#EF934C')
            figures['06_road_distance'].update_yaxes(title='จำนวนแปลง')
        notes.append(f'ระยะทางไม่มีค่า {len(d)-len(road):,} แปลง; ไม่แทนด้วย 0 กม. '
                     'ระยะปัจจุบันยังไม่รวมการเดินทางจากแปลงไป node ถนน')
    else:
        notes.append('ไม่มีข้อมูลระยะทาง: ข้าม histogram ถนน ให้ตรวจ H')

    for fig in figures.values():
        fig.update_layout(template='plotly_white', paper_bgcolor='white', plot_bgcolor='white',
            font=dict(family='Sarabun, Tahoma, sans-serif', size=13, color='#24313B'),
            height=470, margin=dict(l=60, r=25, t=100, b=70),
            legend=dict(orientation='h', yanchor='bottom', y=1.01, x=0),
            title=dict(font=dict(size=19, color='#AD2926'), x=0.02))
    return figures, tables, notes


# ไม่ใช้ข้อมูลจำลอง: ถ้ามีผลล่าสุดใน memory ให้ใช้; ถ้าเปิดใหม่อ่าน CSV จาก J
chart_plots = master_plots if 'master_plots' in globals() else pd.read_csv(
    'burnwise_master_plots.csv', dtype={'tambon_id': str})
chart_overview = df_tambon_overview if 'df_tambon_overview' in globals() else (
    pd.read_csv('burnwise_tambon_overview.csv') if Path('burnwise_tambon_overview.csv').is_file() else None)
chart_figures, chart_tables, chart_notes = make_burnwise_charts(
    chart_plots, chart_overview,
    burn_threshold=globals().get('BURN_PLOT_THRESHOLD', 10.0),
    coverage_threshold=globals().get('MIN_VALID_OBSERVATION_PCT', 80.0))

chart_summary = pd.DataFrame({
    'รายการ': ['แปลงทั้งหมด', 'ตรวจพบสัญญาณเผา', 'ไม่เข้าเกณฑ์ตรวจพบ', 'ข้อมูลไม่เพียงพอ'],
    'จำนวนแปลง': [len(chart_plots)] + [int(chart_plots['burn_status'].eq(s).sum()) for s in STATUS_ORDER]})
display(chart_summary)
for name, fig in chart_figures.items():
    fig.show(renderer="browser")
for note in chart_notes:
    print('หมายเหตุ:', note)

# HTML รวมเปิดดูได้ออฟไลน์ + JSON สำหรับใช้กับ Plotly/Streamlit ในเว็บ
from html import escape as html_escape
CHART_DIR = Path('burnwise_charts')
CHART_DIR.mkdir(exist_ok=True)
sections = []
for i, (name, fig) in enumerate(chart_figures.items()):
    sections.append(pio.to_html(fig, full_html=False, include_plotlyjs=(i == 0),
        config={'responsive': True, 'displaylogo': False}, div_id=name))
    (CHART_DIR / f'{name}.json').write_text(fig.to_json(), encoding='utf-8')
for name, frame in chart_tables.items():
    frame.to_csv(CHART_DIR / f'{name}.csv', index=False, encoding='utf-8-sig')
notes_html = ''.join(f'<li>{html_escape(s)}</li>' for s in chart_notes)
html_doc = '<!doctype html><html lang="th"><head><meta charset="utf-8">' \
    '<meta name="viewport" content="width=device-width,initial-scale=1">' \
    '<title>BurnWise · กราฟผลวิเคราะห์</title><style>' \
    'body{margin:0;background:#fff7f1;color:#24313B;font-family:Tahoma,sans-serif}' \
    'main{max-width:1200px;margin:auto;padding:24px}h1{color:#AD2926}' \
    '.chart{background:white;border:1px solid #eaded6;border-radius:16px;margin:20px 0;padding:12px}' \
    'li{line-height:1.8}</style></head><body><main><h1>BurnWise · กราฟผลวิเคราะห์</h1>' \
    '<p>สัญญาณจากแบบจำลอง ยังไม่ใช่การยืนยันเผาจริง; No Burn ไม่ได้ยืนยันการไถกลบ</p>' \
    + ''.join('<section class="chart">' + s + '</section>' for s in sections) \
    + '<section class="chart"><h2>ฐานข้อมูลและข้อจำกัด</h2><ul>' + notes_html \
    + '</ul></section></main></body></html>'
(CHART_DIR / 'burnwise_charts.html').write_text(html_doc, encoding='utf-8')
print('บันทึกกราฟแล้ว:', (CHART_DIR / 'burnwise_charts.html').resolve())
print('JSON แต่ละกราฟนำไปอ่านด้วย plotly.io.from_json และแสดงด้วย st.plotly_chart ในเว็บได้')
print('สูตรภาพและข้อมูล CSV ที่โหลดต้องเป็นรอบเดียวกัน; ไม่มีการคำนวณ GEE ใหม่ในเซลล์นี้')


## O2 · เปรียบเทียบกับผลิตภัณฑ์ GISTDA ภายนอก (ส่วนเพิ่มเติม)
ส่วนหลักส่งออกได้ก่อนส่วนนี้ ต้องมี reference ครบทุกเดือน ไม่มีช่องว่าง และ raster grid ที่ export ด้วยสูตรเดียวกัน ไม่เปลี่ยนช่วงภาพเพียงเพื่อให้คะแนนดีขึ้น

ขั้นตอน: วาง shapefile ใน `gistda/` → รัน O2.1–2 → ตั้ง `START_GRID_EXPORT=True` ใน O2.3 เพื่อส่งงาน → เมื่อ task เสร็จดาวน์โหลด tif จาก Drive/BurnWise → รัน O2.4–5

เปรียบเทียบบน **pixel เกษตรที่ภาพใช้ได้ชุดเดียวกัน** ของทั้งสองผลิตภัณฑ์ ไม่ใช้ตัวหารพื้นที่แปลงคนละชนิด; นอก polygon GISTDA หมายถึงผลิตภัณฑ์ไม่ตรวจพบ ไม่ใช่ยืนยันไม่เผา

GISTDA กับ BurnWise ใช้ Sentinel-2 ร่วมกัน อาจมีข้อผิดพลาดสัมพันธ์กัน ผลคือ product agreement; plot_label_IoU เป็น IoU ของชุดแปลงที่ถูกจัดเป็น Burn ไม่ใช่ spatial IoU ของพื้นที่ไหม้

### O2.1 · ไฟล์และช่วงเวลาครบหรือไม่

In [ ]:
from pathlib import Path
import os,re,json,hashlib
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
# ระบุโฟลเดอร์ข้อมูลที่แตก zip แล้ว ไม่ค้นทั้ง Downloads เพื่อป้องกันไฟล์ซ้ำ/ผิดเดือน
GISTDA_FOLDER=Path('gistda')
GISTDA_LOCAL_PATHS=list(GISTDA_FOLDER.rglob('Burned_S2_*.shp'))
if not GISTDA_LOCAL_PATHS:
    raise FileNotFoundError('วางไฟล์ Burned_S2 ของ พ.ย.2025 ธ.ค.2025 ม.ค.2026 พร้อม .dbf/.shx/.prj ในโฟลเดอร์ gistda แล้วรันส่วน O2 ใหม่; ส่วนหลัก A–N ใช้ต่อได้')
unique={}
for p in GISTDA_LOCAL_PATHS:
    unique[os.path.normcase(os.path.realpath(p))]=p
GISTDA_LOCAL_PATHS=list(unique.values())
periods=[]
for p in GISTDA_LOCAL_PATHS:
    for ext in ['.dbf','.shx','.prj']:
        if not p.with_suffix(ext).is_file():raise FileNotFoundError(f'{p} ขาด {ext}')
    m=re.search(r'(\d{8})_(\d{8})',p.name)
    if not m:raise ValueError(f'ระบุวันไม่ได้จาก {p.name} ต้องตรวจ metadata ก่อน')
    periods.append((pd.to_datetime(m[1],format='%Y%m%d'),pd.to_datetime(m[2],format='%Y%m%d')+pd.Timedelta(days=1),p))
# เลือกเฉพาะไฟล์ใน post และตรวจ coverage แบบต่อเนื่อง ไม่ใช้ min/max ปิดบังเดือนที่หาย
start,end=pd.Timestamp(post_start),pd.Timestamp(post_end)
periods=[v for v in periods if v[0]<end and v[1]>start]
if any(s<start or e>end for s,e,p in periods):
    raise ValueError('ไฟล์ reference ครอบคลุมเกิน post และไม่มีวันราย polygon ให้กรอง ต้องเลือกช่วงที่ตรงกัน')
cursor=start
for s,e,p in sorted(periods,key=lambda x:x[0]):
    if s>cursor:raise ValueError(f'GISTDA ขาดข้อมูลตั้งแต่ {cursor.date()} ถึงก่อน {s.date()} — เพิ่มเดือนที่ขาดก่อนประเมิน')
    cursor=max(cursor,e)
if cursor<end:raise ValueError(f'GISTDA ครอบคลุมถึงก่อน {cursor.date()} แต่ต้องถึงก่อน {end.date()} — เพิ่มไฟล์ที่ขาดก่อนประเมิน')
GISTDA_LOCAL_PATHS=[p for s,e,p in periods]
GISTDA_VALIDATION_START=post_start;GISTDA_VALIDATION_END=post_end
GISTDA_DATE_FIELD=None;GISTDA_DATE_FILTERED=False
BURNWISE_PLOT_THRESHOLD=BURN_PLOT_THRESHOLD
GISTDA_OVERLAP_THRESHOLD=0.10
EXCLUDE_BOUNDARY_FRAGMENTS=True
print('reference ครบช่วง:',post_start,'ถึงก่อน',post_end)
for p in GISTDA_LOCAL_PATHS:print(p)

### O2.2 · อ่าน reference ตาม CRS ที่กำกับ

In [ ]:
import fiona
from rasterio.warp import transform_bounds
parts=[]
aoi_4326=gdf_tambons.to_crs('EPSG:4326')
for p in GISTDA_LOCAL_PATHS:
    with fiona.open(p) as meta:
        source_crs=meta.crs_wkt or meta.crs
    if not source_crs:raise ValueError(f'{p} ไม่มี CRS ต้องตรวจแหล่งข้อมูลก่อน')
    bbox=transform_bounds('EPSG:4326',source_crs,*aoi_4326.total_bounds,densify_pts=21)
    g=gpd.read_file(p,bbox=bbox).to_crs(EVAL_CRS)
    g['geometry']=g.geometry.make_valid()
    g=g[g.geom_type.isin(['Polygon','MultiPolygon']) & ~g.geometry.is_empty]
    g['gistda_file']=p.name
    parts.append(g)
gistda_val=gpd.GeoDataFrame(pd.concat(parts,ignore_index=True),crs=EVAL_CRS)
gistda_val=gpd.clip(gistda_val,aoi_4326.to_crs(EVAL_CRS)).reset_index(drop=True)
gistda_val['geometry']=gistda_val.geometry.make_valid()
gistda_val=gistda_val[gistda_val.geom_type.isin(['Polygon','MultiPolygon']) & ~gistda_val.geometry.is_empty].reset_index(drop=True)
gistda_val['gistda_id']=range(len(gistda_val))
print('polygon reference ใน AOI:',len(gistda_val))
print('เป็นผลิตภัณฑ์จากดาวเทียมภายนอก ไม่ใช่ ground truth ภาคสนาม; นอก polygon คือไม่มี reference ตรวจพบ')

### O2.3 · Export grid กลาง (ทำเมื่อสูตร/วันที่เปลี่ยน)

In [ ]:
# ต้องใช้ grid เดียวกันเพื่อประเมินทั้ง BurnWise และ GISTDA บนพื้นที่เกษตรที่มีภาพใช้ได้
# ตั้ง True เพื่อส่งงาน Export แล้วเปิด Google Drive ดาวน์โหลด tif พร้อม manifest กลับมาข้าง notebook
START_GRID_EXPORT = True
grid_config={'pre_start':pre_start,'pre_end':pre_end,'post_start':post_start,'post_end':post_end,
    'threshold':float(THRESHOLD),'ndvi_pre_min':NDVI_PRE_MIN,'ndvi_post_max':NDVI_POST_MAX,
    'crs':EVAL_CRS,'transform':EVAL_TRANSFORM,'worldcover':'ESA/WorldCover/v100',
    'aoi_sha256':hashlib.sha256(Path(geojson_path).read_bytes()).hexdigest()}
grid_key=hashlib.sha256(json.dumps(grid_config,sort_keys=True).encode()).hexdigest()[:16]
grid_prefix=f'burnwise_evaluation_grid_{grid_key}'
GRID_PATH=Path(grid_prefix+'.tif')
GRID_MANIFEST=Path(grid_prefix+'_manifest.json')
print('ไฟล์ raster ที่ต้องใช้:',GRID_PATH)
if START_GRID_EXPORT:
    grid_image=(cropland_mask.unmask(0).rename('crop')
        .addBands(valid_crop_mask.unmask(0).rename('valid'))
        .addBands(burn_mask.unmask(0).rename('burn')).toUint8().clip(aoi))
    grid_task=ee.batch.Export.image.toDrive(image=grid_image,description='BurnWise_evaluation_grid',
        folder='BurnWise',fileNamePrefix=grid_prefix,region=aoi,
        crs=EVAL_CRS,crsTransform=EVAL_TRANSFORM,maxPixels=1e10,
        fileFormat='GeoTIFF',formatOptions={'cloudOptimized':True})
    grid_task.start()
    GRID_MANIFEST.write_text(json.dumps(grid_config,ensure_ascii=False,indent=2),encoding='utf-8')
    print('เริ่ม Export:',grid_task.id,'เมื่อ COMPLETED ดาวน์โหลด tif จาก Drive/BurnWise กลับมาข้าง notebook')
else:
    print('ใช้ tif เดิมถ้ามีและ manifest ตรงสูตรปัจจุบัน; ถ้ายังไม่มี ตั้ง START_GRID_EXPORT=True แล้วรันเซลล์นี้')

In [ ]:
status = grid_task.status()
print(status.get("state"))
print(status.get("error_message", ""))

In [ ]:
print(grid_task.status())

### O2.4 · ประเมินทั้งสองผลิตภัณฑ์บน grid เดียวกัน

In [ ]:
import rasterio
from rasterio.features import rasterize,geometry_mask,geometry_window
from rasterio.errors import WindowError
if not GRID_PATH.is_file() or not GRID_MANIFEST.is_file():
    raise FileNotFoundError('ต้องมี burnwise_evaluation_grid.tif และ manifest จากเซลล์ Export ก่อน; ไม่ต้องรันส่วนหลักใหม่')
if json.loads(GRID_MANIFEST.read_text(encoding='utf-8'))!=grid_config:
    raise ValueError('raster/manifest เป็นสูตรหรือช่วงเวลาเก่า ต้อง Export ใหม่ก่อนประเมิน')
plot_geom=plots_all[['plot_id','geometry']].to_crs(EVAL_CRS)
assert plot_geom['plot_id'].is_unique and df_plots['plot_id'].is_unique
plots_val=plot_geom.merge(df_plots,on='plot_id',how='inner',validate='one_to_one')
use=plots_val['exclusion_reason'].isna() & plots_val['burn_pct'].notna()
if EXCLUDE_BOUNDARY_FRAGMENTS:use &= ~plots_val['is_boundary_fragment'].fillna(False).astype(bool)
plots_val['validation_used']=False
eligible=plots_val.loc[use].copy()
# spatial index เลือก reference เฉพาะ window; ไม่ rasterize ทั้งประเทศไทย
results=[]
with rasterio.open(GRID_PATH) as ds:
    if ds.crs.to_epsg()!=32647 or ds.count!=3 or ds.res!=(10.0,10.0):
        raise ValueError('raster ต้องเป็น UTM47N, 3 bands crop/valid/burn, 10 เมตร')
    if abs(ds.transform.c/ds.res[0]-round(ds.transform.c/ds.res[0]))>1e-6 or abs(ds.transform.f/ds.res[1]-round(ds.transform.f/ds.res[1]))>1e-6:
        raise ValueError('raster ไม่ตรง grid origin ที่ตั้งไว้')
    for i,row in enumerate(eligible.itertuples()):
        try:w=geometry_window(ds,[row.geometry.__geo_interface__])
        except WindowError:continue
        arr=ds.read(window=w,masked=True)
        trans=ds.window_transform(w)
        inside=geometry_mask([row.geometry.__geo_interface__],arr.shape[1:],trans,invert=True,all_touched=False)
        crop=inside & ~np.ma.getmaskarray(arr[0]) & (arr[0].filled(0)==1)
        valid=crop & ~np.ma.getmaskarray(arr[1]) & (arr[1].filled(0)==1)
        nc=int(crop.sum());nv=int(valid.sum())
        if nc==0 or nv/nc*100 < MIN_VALID_OBSERVATION_PCT:continue
        burned=valid & (arr[2].filled(0)==1)
        ids=list(gistda_val.sindex.query(row.geometry,predicate='intersects'))
        ref=np.zeros(inside.shape,dtype=np.uint8)
        if ids:
            shapes=[(g,1) for g in gistda_val.iloc[ids].geometry if not g.is_empty]
            ref=rasterize(shapes,out_shape=inside.shape,transform=trans,fill=0,dtype='uint8',all_touched=False)
        bw_pct=100*burned.sum()/nv
        ref_pct=100*((ref==1)&valid).sum()/nv
        results.append({'plot_id':row.plot_id,'BurnWise_Burn':int(bw_pct>=BURNWISE_PLOT_THRESHOLD),
            'GISTDA_Burn':int(ref_pct>=GISTDA_OVERLAP_THRESHOLD*100),
            'burnwise_eval_pct':bw_pct,'gistda_overlap_pct':ref_pct,'eval_pixels':nv})
        if i%1000==0:print('ประเมิน',i,'/',len(eligible),flush=True)
if not results:raise ValueError('ไม่มีแปลงผ่าน coverage สำหรับ validation')
plots_eval=eligible.merge(pd.DataFrame(results),on='plot_id',validate='one_to_one')
plots_eval['validation_used']=True
plots_val['validation_used']=plots_val['plot_id'].isin(plots_eval['plot_id'])
for name,yp,yt in [('TP',1,1),('FP',1,0),('FN',0,1),('TN',0,0)]:
    plots_eval[name]=((plots_eval['BurnWise_Burn']==yp)&(plots_eval['GISTDA_Burn']==yt)).astype(int)
y_true=plots_eval['GISTDA_Burn'];y_pred=plots_eval['BurnWise_Burn']
cm=confusion_matrix(y_true,y_pred,labels=[0,1])
tn,fp,fn,tp=cm.ravel()
def metric_row(g):
    yt=g['GISTDA_Burn'];yp=g['BurnWise_Burn'];a,b,c,d=confusion_matrix(yt,yp,labels=[0,1]).ravel()
    return {'n_plots':len(g),'TN':int(a),'FP':int(b),'FN':int(c),'TP':int(d),
        'Accuracy':accuracy_score(yt,yp),'Precision':precision_score(yt,yp,zero_division=np.nan),
        'Recall':recall_score(yt,yp,zero_division=np.nan),'F1':f1_score(yt,yp,zero_division=np.nan),
        'plot_label_IoU':float(d/(b+c+d)) if b+c+d else np.nan}
validation_metrics=pd.DataFrame([{**metric_row(plots_eval),'comparison_type':'external_product_agreement',
    'validation_start':post_start,'validation_end_exclusive':post_end,
    'support':'same valid cropland pixels; 10m UTM grid; pixel centers',
    'n_plots_total':len(plots_val),'n_excluded':len(plots_val)-len(plots_eval),
    'reference_files':';'.join(str(p) for p in GISTDA_LOCAL_PATHS)}])
print('ความสอดคล้องกับผลิตภัณฑ์ GISTDA ไม่ใช่ความแม่นยำต่อหลักฐานภาคสนาม')
display(validation_metrics)
print('Confusion rows=GISTDA product, columns=BurnWise');print(cm)

### O2.5 · ส่งออกผลเปรียบเทียบ

In [ ]:
rows=[]
for (tid,tname),g in plots_eval.groupby(['tambon_id','tambon_name']):
    rows.append({'tambon_id':tid,'tambon_name':tname,**metric_row(g)})
tambon_validation=pd.DataFrame(rows)
validation_metrics.to_csv('burnwise_gistda_validation_metrics.csv',index=False,encoding='utf-8-sig')
tambon_validation.to_csv('burnwise_gistda_validation_tambon.csv',index=False,encoding='utf-8-sig')
plots_eval.drop(columns='geometry').to_csv('burnwise_gistda_validation_plots.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(cm,index=['GISTDA_not_detected','GISTDA_detected'],columns=['BurnWise_not_detected','BurnWise_detected']).to_csv('burnwise_gistda_confusion_matrix.csv',encoding='utf-8-sig')
display(tambon_validation)

## วิธีใช้ผลและข้อจำกัด
- ผล Burn/No Burn เป็นเกณฑ์ตรวจพบสัญญาณ ยังไม่ยืนยันการเผาจริงหรือการไถกลบ
- pre/post median ช่วงละ 3 เดือนอาจกลบร่องรอยระยะสั้น; ภาพ pre น้อยต้องตรวจคุณภาพและเลือกช่วงตามเหตุการณ์จริง
- threshold เป็นค่าเฉลี่ย dNBR ของ hotspot calibration ไม่ใช่ threshold ที่ผ่านการ optimize กับสอง class; จุด test แยกตัวอย่างแต่ไม่รับประกันอิสระเชิงพื้นที่/เหตุการณ์
- ขอบเขต FTW เป็นผลโมเดล การ assign ตำบลด้วย centroid ไม่ได้ตัดแปลงทุกแนวเขต และจำนวนแปลงต่างจากเวอร์ชันตัดรายตำบลได้
- shapefile เดือนเดียวไม่รองรับ validation ทั้งสามเดือน ไม่ควรตีความ FP/TN ว่าเป็นความจริงภาคสนาม
- รายได้เฉลี่ยตำบลเป็นบริบทพื้นที่ ถนนอาจไม่รวมทางเข้าแปลง จุดรับซื้อยังต้องตรวจสอบ
- Dashboard รุ่นเก่าใช้ tier/green_score/tillage_pct: ต้องปรับให้ใช้ burn_tier/burn_status/no_burn_pct ก่อนโหลด CSV ชุดนี้ ไม่มีข้อมูลจำลองหรือผลรันเก่าถูกเติมให้ดูครบ

# L · Stage 2: ประเมินกับชุดอ้างอิงอิสระ (Ground Truth)

⚠️ **ส่วนนี้รันไม่ได้จนกว่าจะมีไฟล์ `reference_labels.csv` ที่กรอกข้อมูลจริงแล้ว** (ดู `reference_labels_template.csv` เป็นโครงคอลัมน์) — ต้องเก็บจากการสำรวจภาคสนามหรือหลักฐานภาพ+วันที่จริง **ห้ามใช้จุด VIIRS/FIRMS ชุดเดิมที่ใช้ตั้ง THRESHOLD มาใส่ซ้ำ** เพราะจะเป็นการ validate กับข้อมูลที่ใช้ calibrate ไปแล้ว (circular)

คอลัมน์ที่ต้องมี: `reference_id, plot_id, event_date, label (burn/no_burn/uncertain), evidence_source, evidence_date, confidence, is_full_plot, notes`

In [ ]:
from pathlib import Path
import pandas as pd

REF_LABELS_PATH = Path('reference_labels.csv')  # แก้ path ตามตำแหน่งไฟล์จริง

if not REF_LABELS_PATH.is_file():
    print('❌ ยังไม่พบ reference_labels.csv — หยุดตรงนี้ก่อน ต้องเก็บข้อมูลภาคสนามให้ครบตามเทมเพลตก่อน')
    print('   ดู reference_labels_template.csv เป็นแนวทางคอลัมน์ที่ต้องมี')
else:
    ref = pd.read_csv(REF_LABELS_PATH)
    required = {'reference_id', 'plot_id', 'label', 'evidence_source', 'confidence'}
    missing = required - set(ref.columns)
    if missing:
        raise ValueError(f'reference_labels.csv ขาดคอลัมน์: {sorted(missing)}')
    print(f'โหลด reference labels สำเร็จ: {len(ref)} รายการ')
    print(ref['label'].value_counts())

## L.1 เช็คความอิสระ (Independence Check)

เตือนถ้า `plot_id` ในชุดอ้างอิงไปซ้ำกับแปลงที่มีจุด VIIRS Calibration (`calib_ids`) ที่ใช้ตั้งค่า THRESHOLD อยู่แล้ว — ถ้าซ้ำ แปลว่าไม่ใช่การทดสอบอิสระจริง ต้องตัดออกจากการคำนวณ metric หรือหาแปลงอ้างอิงใหม่แทน

In [ ]:
if 'ref' in dir() and 'calib_ids' in dir():
    # หา plot_id ที่ตรงกับจุดที่อยู่ใน fire_points ชุด calibration (เทียบจาก centroid ใกล้ที่สุด)
    calib_plot_ids = set()
    try:
        calib_pts_info = fire_points.filter(ee.Filter.inList('system:index', calib_ids)).geometry().getInfo()
        # หมายเหตุ: การเทียบ plot_id โดยตรงทำได้แม่นกว่าถ้า fire_points มี property plot_id ติดมาด้วย
        # ถ้าไม่มี ให้ข้ามเช็คนี้และบันทึกไว้เป็นข้อจำกัดในรายงานแทน
    except Exception as e:
        print(f'เช็คอัตโนมัติไม่สำเร็จ ({e}) — ให้ตรวจด้วยมือว่า reference_id ไม่ซ้ำกับจุดที่ใช้ตั้ง threshold')

    overlap = ref[ref['plot_id'].isin(calib_plot_ids)]
    if len(overlap) > 0:
        print(f'⚠️ พบ {len(overlap)} รายการในชุดอ้างอิงที่ plot_id ซ้ำกับจุด Calibration — ตัดออกจากการประเมินด้านล่าง')
        ref = ref[~ref['plot_id'].isin(calib_plot_ids)]
    else:
        print('✅ ไม่พบ plot_id ซ้ำกับชุด Calibration อัตโนมัติ (ยังควรตรวจด้วยมืออีกรอบก่อนเชื่อ 100%)')

## L.2 รวมผลกับ df_plots แล้วคำนวณ Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

if 'ref' in dir():
    ref_known = ref[ref['label'].isin(['burn', 'no_burn'])].copy()
    n_uncertain = (ref['label'] == 'uncertain').sum()
    print(f'ตัดรายการ uncertain ออก {n_uncertain} รายการ — ไม่ใช้คำนวณ metric')

    eval_df = ref_known.merge(
        df_plots[['plot_id', 'burn_status', 'tambon_name']], on='plot_id', how='left'
    )
    n_missing = eval_df['burn_status'].isna().sum()
    if n_missing > 0:
        print(f'⚠️ {n_missing} รายการหา plot_id ไม่เจอใน df_plots — ตรวจรหัสแปลงให้ตรงกันก่อน')
        eval_df = eval_df.dropna(subset=['burn_status'])

    eval_df = eval_df[eval_df['burn_status'] != 'ข้อมูลไม่เพียงพอ']
    n_insufficient = len(ref_known) - len(eval_df)
    print(f'ตัดแปลงที่โมเดลให้สถานะ "ข้อมูลไม่เพียงพอ" ออก {n_insufficient} รายการ — รายงานแยกต่างหาก ไม่ใช้คำนวณ accuracy')

    if len(eval_df) == 0:
        print('❌ ไม่เหลือข้อมูลพอคำนวณ metric หลังกรอง')
    else:
        y_true = (eval_df['label'] == 'burn').astype(int)
        y_pred = (eval_df['burn_status'] == 'Burn').astype(int)

        cm = confusion_matrix(y_true, y_pred)
        print('Confusion Matrix (แถว=จริง, คอลัมน์=โมเดลทำนาย) [[TN,FP],[FN,TP]]:')
        print(cm)

        n = len(eval_df)
        print(f'\nจำนวนตัวอย่างที่ประเมินได้จริง: {n} รายการ')
        if n < 30:
            print('⚠️ ตัวอย่างน้อยกว่า 30 รายการ — ตัวเลข precision/recall/F1 ด้านล่างเป็นสัญญาณเบื้องต้นเท่านั้น '
                  'ยังสรุปภาพรวมความแม่นยำของทั้งโครงการไม่ได้')

        print(f'Accuracy:  {accuracy_score(y_true, y_pred):.3f}')
        print(f'Precision: {precision_score(y_true, y_pred, zero_division=0):.3f}')
        print(f'Recall:    {recall_score(y_true, y_pred, zero_division=0):.3f}')
        print(f'F1:        {f1_score(y_true, y_pred, zero_division=0):.3f}')

## L.3 สรุปรายตำบล (ถ้าจำนวนตัวอย่างพอ)

In [ ]:
if 'eval_df' in dir() and len(eval_df) >= 10:
    by_tambon = eval_df.groupby('tambon_name').apply(
        lambda g: pd.Series({
            'n': len(g),
            'accuracy': accuracy_score((g['label']=='burn').astype(int), (g['burn_status']=='Burn').astype(int))
                        if len(g) >= 5 else None,
        })
    )
    print(by_tambon)
    print('\nตำบลที่มีตัวอย่างน้อยกว่า 5 รายการ ไม่แสดง accuracy แยก เพราะตัวอย่างน้อยเกินสรุป')
else:
    print('ตัวอย่างรวมน้อยกว่า 10 รายการ ยังไม่แบ่งราย ตำบล — เก็บข้อมูลเพิ่มก่อน')

# M · Sentinel-1 SAR Roughness + SRTM Slope (ฟีเจอร์เสริมแยกความกำกวมเผา/ไถกลบ)

ตามคำแนะนำจากกรรมการ/รุ่นพี่: dNBR อย่างเดียวแยก **รอยเผา (เถ้าถ่านเรียบ)** กับ **ดินไถกลบ (ผิวขรุขระ)** ได้ไม่ชัดพอ เพราะสเปกตรัมทับซ้อนกันสูง ส่วนนี้ดึง **Sentinel-1 SAR backscatter (VV)** มาเป็นสัญญาณเสริม — ดินไถกลบมักสะท้อนสัญญาณเรดาร์แรงกว่า (ขรุขระ) ส่วนเถ้าถ่านเรียบมักสะท้อนอ่อนกว่า

เพิ่ม **SRTM Slope** ด้วย ตามคำแนะนำเรื่อง Access Gap — พื้นที่ลาดชันคือข้อจำกัดทางกายภาพจริงที่รถไถเข้าไม่ถึง

⚠️ **หมายเหตุสำคัญ:** ส่วนนี้ให้ตัวเลข `sar_vv_diff` และ `mean_slope_deg` เป็น **ฟีเจอร์เสริม/ข้อมูลประกอบการตัดสินใจ** เท่านั้น ยังไม่ได้ใช้เปลี่ยนตรรกะ `burn_status` หลัก เพราะการเปลี่ยน threshold/เกณฑ์หลักตอนนี้ต้องรัน Stage 1-2 ใหม่ทั้งชุดและเสี่ยงเวลาไม่พอ — ใช้เป็นคอลัมน์ไว้ตรวจแปลงที่ "ก้ำกึ่ง"/Unknown เพิ่มเติมได้ก่อน แล้วค่อยพิจารณาผนวกเข้าสูตรหลักในรอบถัดไป

In [ ]:
S1_VV_BANDS = ['VV']

def get_s1_median(start, end, aoi_geom):
    coll = (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(aoi_geom)
            .filterDate(start, end)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
            .select('VV'))
    n = coll.size().getInfo()
    return coll.median(), n

s1_pre, n_s1_pre = get_s1_median(pre_start, pre_end, aoi)
s1_post, n_s1_post = get_s1_median(post_start, post_end, aoi)
print(f'ภาพ Sentinel-1 ช่วงก่อน: {n_s1_pre} ภาพ, ช่วงหลัง: {n_s1_post} ภาพ')

if n_s1_pre == 0 or n_s1_post == 0:
    print('⚠️ ไม่มีภาพ Sentinel-1 เพียงพอในช่วงที่เลือก — ข้ามส่วน SAR นี้ไปก่อน (ไม่กระทบ pipeline หลัก)')
    SKIP_SAR = True
else:
    SKIP_SAR = False
    s1_vv_diff = s1_post.subtract(s1_pre).rename('sar_vv_diff')
    s1_stats_image = (s1_pre.rename('sar_vv_pre')
                       .addBands(s1_post.rename('sar_vv_post'))
                       .addBands(s1_vv_diff))
    print('สร้างภาพ SAR VV (pre/post/diff) สำเร็จ')

In [ ]:
dem = ee.Image('USGS/SRTMGL1_003')
slope_img = ee.Terrain.slope(dem).rename('mean_slope_deg')
print('โหลด SRTM + คำนวณ slope สำเร็จ')

In [ ]:
import geemap

assert 'ftw_aoi' in dir(), 'ไม่พบ ftw_aoi — ต้องรันส่วน F (FTW) ให้เสร็จก่อนรันส่วนนี้'

combine_bands = slope_img if SKIP_SAR else slope_img.addBands(s1_stats_image)

sar_slope_rows = []
batch_size = 500
plot_geoms = ftw_aoi[['plot_id', 'geometry']].reset_index(drop=True)

for start in range(0, len(plot_geoms), batch_size):
    batch_gdf = plot_geoms.iloc[start:start+batch_size]
    ee_fc = geemap.geopandas_to_ee(batch_gdf)
    stats = combine_bands.reduceRegions(
        collection=ee_fc, reducer=ee.Reducer.mean(), scale=10, tileScale=16
    ).getInfo()
    for f in stats['features']:
        p = f['properties']
        row = {
            'plot_id': p.get('plot_id'),
            'mean_slope_deg': p.get('mean_slope_deg'),
        }
        if not SKIP_SAR:
            row['sar_vv_pre'] = p.get('sar_vv_pre')
            row['sar_vv_post'] = p.get('sar_vv_post')
            row['sar_vv_diff'] = p.get('sar_vv_diff')
        sar_slope_rows.append(row)
    print(f'ประมวลผลแล้ว {min(start+batch_size, len(plot_geoms))}/{len(plot_geoms)} แปลง')

df_sar_slope = pd.DataFrame(sar_slope_rows)
print(f'\nสร้างตาราง SAR/Slope สำเร็จ: {len(df_sar_slope)} แปลง')
df_sar_slope.head()

## M.1 บันทึกและวิธีรวมเข้ากับตารางหลัก

บันทึกแยกเป็นไฟล์ของตัวเองก่อน แล้วค่อย **LEFT JOIN** กับ `master_plots` ด้วย `plot_id` — ไม่เขียนทับ `master_plots` ตรงๆ ในรอบนี้ เพื่อให้ตรวจสอบผลใหม่ก่อนรวมจริง

In [ ]:
df_sar_slope.to_csv('burnwise_plots_sar_slope.csv', index=False, encoding='utf-8-sig')
print('บันทึก burnwise_plots_sar_slope.csv แล้ว')

try:
    from google.colab import files as _colab_files
    _colab_files.download('burnwise_plots_sar_slope.csv')
except ImportError:
    pass

# ตัวอย่างวิธีรวมกับ master_plots (รันเองอีกทีหลังตรวจผลแล้ว):
# master_plots_v2 = master_plots.merge(df_sar_slope, on='plot_id', how='left')
print('\n>>> วิธีรวม: master_plots_v2 = master_plots.merge(df_sar_slope, on="plot_id", how="left") <<<')

## O3 · เปรียบเทียบกับผลิตภัณฑ์ HII "ตามรอยเผา" (อิสระจาก GISTDA)

O2 เทียบกับ GISTDA ซึ่งใช้ Sentinel-2 เหมือนกับเรา O3 นี้เทียบกับ **สถาบันสารสนเทศทรัพยากรน้ำ (HII)** คนละหน่วยงาน คนละทีมพัฒนา — ให้สัญญาณอิสระกว่าการเทียบกับ GISTDA เพียงอย่างเดียว

**ขั้นตอนเตรียมไฟล์ (ทำนอกโน้ตบุ๊ก):**
1. เข้า https://tamroypao.hii.or.th/ → เลือกผลิตภัณฑ์ B4 (Level 4 Burn Scar, รายสัปดาห์, 20×20m)
2. ดาวน์โหลด GeoTIFF ของสัปดาห์ที่ครอบคลุมช่วง พ.ย. 2025 – ม.ค. 2026 (ตรงกับ `post_start`/`post_end` ที่ใช้ทั้งไฟล์)
3. วางไฟล์ `.tif` ทั้งหมดไว้ในโฟลเดอร์ `hii/`

⚠️ **ข้อควรระวังเดียวกับ O2:** HII ก็เป็นผลิตภัณฑ์จากดาวเทียมเช่นกัน ไม่ใช่หลักฐานภาคสนาม ผลลัพธ์คือ **product agreement** ไม่ใช่ accuracy ต่อความจริง — ใช้คำนี้เท่านั้นเวลารายงาน ห้ามเรียกว่า "ความแม่นยำ"

In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
from pathlib import Path

HII_FOLDER = Path('hii')
HII_LOCAL_PATHS = sorted(HII_FOLDER.rglob('*.tif'))
if not HII_LOCAL_PATHS:
    raise FileNotFoundError(
        'วางไฟล์ GeoTIFF จาก tamroypao.hii.or.th (ผลิตภัณฑ์ B4) ในโฟลเดอร์ hii/ ก่อนรันส่วนนี้ — '
        'ส่วน A-O2 และ L, M ใช้ต่อได้ตามปกติโดยไม่ต้องมีไฟล์นี้'
    )
print(f'พบไฟล์ HII {len(HII_LOCAL_PATHS)} ไฟล์:')
for p in HII_LOCAL_PATHS:
    print(' -', p.name)

In [ ]:
# รวมทุกไฟล์ HII ในช่วงที่เลือก เป็น mask เดียว (burn=1 ถ้าสัปดาห์ใดสัปดาห์หนึ่งตรวจพบ) แล้ว reproject ให้ตรงกับ grid ของเราเป๊ะ
# (CRS/transform/resolution ต้องตรงกับ GRID_PATH ที่ export ไว้แล้วใน O2 เพื่อเทียบ pixel ต่อ pixel ได้ตรงกัน)
with rasterio.open(GRID_PATH) as ref_ds:
    ref_crs, ref_transform, ref_w, ref_h = ref_ds.crs, ref_ds.transform, ref_ds.width, ref_ds.height

hii_combined = np.zeros((ref_h, ref_w), dtype='uint8')
for p in HII_LOCAL_PATHS:
    with rasterio.open(p) as src:
        src_band = src.read(1)
        dst = np.zeros((ref_h, ref_w), dtype='uint8')
        reproject(
            source=src_band, destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref_transform, dst_crs=ref_crs,
            resampling=Resampling.nearest,  # ข้อมูล categorical ห้ามใช้ bilinear/cubic
        )
        hii_combined = np.maximum(hii_combined, (dst > 0).astype('uint8'))

print(f'รวม HII burn mask สำเร็จ, พื้นที่ตรวจพบ: {hii_combined.sum()} พิกเซล (10m x 10m ต่อพิกเซล)')

In [ ]:
# ใช้ชุดแปลง eligible ชุดเดียวกับ O2 (ผ่านเกณฑ์ coverage/exclusion_reason แล้ว) ไม่สร้างเกณฑ์คัดกรองใหม่
assert 'eligible' in dir(), 'ต้องรันส่วน O2 (เซลล์สร้าง eligible) ให้เสร็จก่อน เพื่อใช้ชุดแปลงเดียวกัน เทียบกันได้ตรงฐาน'

hii_results = []
with rasterio.open(GRID_PATH) as ds:
    for i, row in enumerate(eligible.itertuples()):
        try:
            w = geometry_window(ds, [row.geometry.__geo_interface__])
        except WindowError:
            continue
        arr = ds.read(window=w, masked=True)
        trans = ds.window_transform(w)
        inside = geometry_mask([row.geometry.__geo_interface__], arr.shape[1:], trans, invert=True, all_touched=False)
        crop = inside & ~np.ma.getmaskarray(arr[0]) & (arr[0].filled(0) == 1)
        valid = crop & ~np.ma.getmaskarray(arr[1]) & (arr[1].filled(0) == 1)
        nc, nv = int(crop.sum()), int(valid.sum())
        if nc == 0 or nv / nc * 100 < MIN_VALID_OBSERVATION_PCT:
            continue
        burned = valid & (arr[2].filled(0) == 1)
        row_off, col_off = w.row_off, w.col_off
        hii_window = hii_combined[row_off:row_off + arr.shape[1], col_off:col_off + arr.shape[2]]
        bw_pct = 100 * burned.sum() / nv
        hii_pct = 100 * (hii_window[:arr.shape[1], :arr.shape[2]][valid] == 1).sum() / nv
        hii_results.append({
            'plot_id': row.plot_id,
            'BurnWise_Burn': int(bw_pct >= BURNWISE_PLOT_THRESHOLD),
            'HII_Burn': int(hii_pct >= GISTDA_OVERLAP_THRESHOLD * 100),
            'burnwise_eval_pct': bw_pct, 'hii_overlap_pct': hii_pct, 'eval_pixels': nv,
        })
        if i % 1000 == 0:
            print('ประเมิน (HII)', i, '/', len(eligible), flush=True)

if not hii_results:
    raise ValueError('ไม่มีแปลงผ่านเกณฑ์ coverage สำหรับเปรียบเทียบกับ HII')

hii_eval = eligible.merge(pd.DataFrame(hii_results), on='plot_id', validate='one_to_one')
print(f'เปรียบเทียบกับ HII ได้ {len(hii_eval)} แปลง')

In [ ]:
def metric_row_hii(g):
    yt, yp = g['HII_Burn'], g['BurnWise_Burn']
    a, b, c, d = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    return {
        'n_plots': len(g), 'TN': int(a), 'FP': int(b), 'FN': int(c), 'TP': int(d),
        'Accuracy': accuracy_score(yt, yp),
        'Precision': precision_score(yt, yp, zero_division=np.nan),
        'Recall': recall_score(yt, yp, zero_division=np.nan),
        'F1': f1_score(yt, yp, zero_division=np.nan),
        'plot_label_IoU': float(d / (b + c + d)) if (b + c + d) else np.nan,
    }

hii_validation_metrics = pd.DataFrame([{
    **metric_row_hii(hii_eval), 'comparison_type': 'external_product_agreement_HII',
    'validation_start': post_start, 'validation_end_exclusive': post_end,
    'support': 'same valid cropland pixels as O2; 10m UTM grid; pixel centers',
    'n_plots_total': len(eligible), 'n_excluded': len(eligible) - len(hii_eval),
    'reference_files': ';'.join(str(p) for p in HII_LOCAL_PATHS),
}])

print('ความสอดคล้องกับผลิตภัณฑ์ HII — ไม่ใช่ความแม่นยำต่อหลักฐานภาคสนาม')
display(hii_validation_metrics)

cm_hii = confusion_matrix(hii_eval['HII_Burn'], hii_eval['BurnWise_Burn'], labels=[0, 1])
print('Confusion rows=HII product, columns=BurnWise')
print(cm_hii)

In [ ]:
rows_hii = []
for (tid, tname), g in hii_eval.groupby(['tambon_id', 'tambon_name']):
    rows_hii.append({'tambon_id': tid, 'tambon_name': tname, **metric_row_hii(g)})
tambon_validation_hii = pd.DataFrame(rows_hii)

hii_validation_metrics.to_csv('burnwise_hii_validation_metrics.csv', index=False, encoding='utf-8-sig')
tambon_validation_hii.to_csv('burnwise_hii_validation_tambon.csv', index=False, encoding='utf-8-sig')
hii_eval.drop(columns='geometry').to_csv('burnwise_hii_validation_plots.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(cm_hii, index=['HII_not_detected', 'HII_detected'],
             columns=['BurnWise_not_detected', 'BurnWise_detected']
             ).to_csv('burnwise_hii_confusion_matrix.csv', encoding='utf-8-sig')

display(tambon_validation_hii)

### O3.1 เทียบ GISTDA vs HII vs BurnWise พร้อมกัน (ถ้ารันทั้ง O2 และ O3 แล้ว)

ถ้าทั้งสองผลิตภัณฑ์เห็นตรงกับเรา = สัญญาณน่าเชื่อถือขึ้นมาก ถ้าเห็นต่างกัน (เช่น GISTDA เห็นแต่ HII ไม่เห็น) = จุดที่ควรเปิดภาพดูเองก่อนสรุป ไม่ใช่เลือกเชื่อฝั่งใดฝั่งหนึ่งเฉยๆ

In [ ]:
if 'plots_eval' in dir() and 'hii_eval' in dir():
    three_way = plots_eval[['plot_id', 'GISTDA_Burn', 'BurnWise_Burn']].merge(
        hii_eval[['plot_id', 'HII_Burn']], on='plot_id', how='inner'
    )
    three_way['all_agree'] = (
        (three_way['GISTDA_Burn'] == three_way['BurnWise_Burn']) &
        (three_way['HII_Burn'] == three_way['BurnWise_Burn'])
    )
    three_way['disagreement'] = ~three_way['all_agree']
    n_agree = three_way['all_agree'].sum()
    n_total = len(three_way)
    print(f'แปลงที่ GISTDA, HII, BurnWise เห็นตรงกันทั้ง 3 แหล่ง: {n_agree}/{n_total} ({n_agree/n_total*100:.1f}%)')

    disagree_plots = three_way[three_way['disagreement']]
    print(f'แปลงที่ยังเห็นไม่ตรงกัน (ควรเปิดภาพตรวจก่อนสรุป): {len(disagree_plots)}')
    three_way.to_csv('burnwise_three_way_comparison.csv', index=False, encoding='utf-8-sig')
    disagree_plots.head(20)
else:
    print('ต้องรันทั้ง O2 (ได้ plots_eval) และ O3 (ได้ hii_eval) ก่อน ถึงจะเทียบ 3 ทางได้')